# 02 — External Features: DEM, ERA5-Land, MODIS NDVI

This notebook must be run after `01_Phase2_Core_Final.ipynb`. Output paths have been standardized; the confusion of `phase2_output`, `phase2_outputs`, and `phase2_outputtt` is not continued in this notebook.

## Security
In some versions of the previous notebook, the NASA Earthdata password and CDS API key were written directly into the cell source code. **Renew/cancel this access information.** The new version contains no secrets. Create the following keys in Colab > Secrets: `NASA_EARTHDATA_USERNAME`, `NASA_EARTHDATA_PASSWORD`, `CDS_API_KEY`. You need to enable notebook access to the secrets. Do not write the password/API key to this file.

## Scientific Notes
- The DEM file is sourced from the Open-Meteo Elevation API; the column name will be `elevation_m`, not `elevation_srtm` unless proven to be SRTM. - In ERA5-Land monthly averages, temperature/wind is weighted with summer daily numbers; `total_precipitation` is converted from the monthly average daily accumulation to the summer total. - MOD13Q1 VI_Quality bits are evaluated according to the product manual. Multiple 16-day observations are expected within the same UrbanId-year; this alone is not a duplicate error. The actual observation duplicate key is at the `UrbanId-Date` level. - NDVI is a point sample from the mapped city coordinate in this notebook; it is not a city polygon average. Results should be interpreted as `centroid/point NDVI proxy`, not as area average.

In [14]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, re, json, time, shutil, zipfile, hashlib, warnings
from pathlib import Path
from io import BytesIO
import numpy as np
import pandas as pd
import requests
import xarray as xr
from datetime import datetime

DRIVE_ROOT = Path('/content/drive/MyDrive')
_REPO_CANDIDATES = [
    DRIVE_ROOT / 'urban-heat-transfer',
    DRIVE_ROOT / 'BIRLESIK_PROJE_PAKETI' / 'urban-heat-transfer',
    DRIVE_ROOT / 'Urban_Heat_Transfer_COMPLETE_BUNDLE' / 'urban-heat-transfer',
]
PROJECT_DIR = next((p for p in _REPO_CANDIDATES if (p / 'notebooks').is_dir() and (p / 'data').is_dir()), None)
LEGACY_LAYOUT = PROJECT_DIR is None
if PROJECT_DIR is None:
    PROJECT_DIR = DRIVE_ROOT / 'tao_documents' if (DRIVE_ROOT / 'tao_documents').is_dir() else DRIVE_ROOT
if LEGACY_LAYOUT:
    PHASE2_ROOT = PROJECT_DIR / 'phase2_outputs'
    EXTERNAL_DIR = PROJECT_DIR / 'external_features'
    BACKUP_ROOT = PROJECT_DIR / 'previous_output_backups'
else:
    PHASE2_ROOT = PROJECT_DIR / 'outputs' / 'phase2_outputs'
    EXTERNAL_DIR = PROJECT_DIR / 'data' / 'external_features'
    BACKUP_ROOT = PROJECT_DIR / 'outputs' / 'previous_output_backups'
WORK_DIR = EXTERNAL_DIR / 'appeears_work'
for p in [PHASE2_ROOT, EXTERNAL_DIR, WORK_DIR]: p.mkdir(parents=True, exist_ok=True)
PRE_RUN_BACKUP_DIR = BACKUP_ROOT / f'external_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
existing_external = [p for p in EXTERNAL_DIR.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv','.md','.json'} and WORK_DIR not in p.parents]
if existing_external:
    for old_path in existing_external:
        rel = old_path.relative_to(PROJECT_DIR)
        dest = PRE_RUN_BACKUP_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(old_path, dest)
    print(f'Archived {len(existing_external):,} existing external-feature files to {PRE_RUN_BACKUP_DIR}')
# One canonical output path across the whole project.
YEAR_MIN, YEAR_MAX = 2001, 2020
YEARS = list(range(YEAR_MIN, YEAR_MAX + 1))
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

MANIFEST_PATH = PHASE2_ROOT / 'analysis_city_manifest.csv'
if not MANIFEST_PATH.exists():
    raise FileNotFoundError('Run 01_Phase2_Core_Final.ipynb first; missing ' + str(MANIFEST_PATH))
manifest = pd.read_csv(MANIFEST_PATH)
need = {'UrbanId','Country','Longitude','Latitude','Area','eligibility_status'}
if not need.issubset(manifest.columns): raise ValueError(f'Manifest missing columns: {need-set(manifest.columns)}')
manifest['UrbanId'] = pd.to_numeric(manifest['UrbanId'], errors='raise').astype(int)
if manifest['UrbanId'].duplicated().any(): raise ValueError('City manifest has duplicate UrbanId rows.')
eligible = manifest[manifest['eligibility_status'].eq('INCLUDED') & manifest['Country'].isin(['China','Türkiye'])].copy()
eligible[['Longitude','Latitude','Area']] = eligible[['Longitude','Latitude','Area']].apply(pd.to_numeric, errors='coerce')
if eligible[['Longitude','Latitude']].isna().any().any(): raise ValueError('Eligible city metadata has missing coordinates.')
if eligible['UrbanId'].duplicated().any(): raise ValueError('Eligible manifest is not unique by UrbanId.')

DEM_PATH = EXTERNAL_DIR / 'dem_elevation.csv'
ERA5_NC_PATH = PHASE2_ROOT / 'era5_land_jja_2001_2020.nc'
ERA5_CSV_PATH = PHASE2_ROOT / 'era5_jja_features.csv'
ERA5_METADATA_PATH = PHASE2_ROOT / 'era5_land_jja_2001_2020_metadata.json'
NDVI_RAW_PATH = EXTERNAL_DIR / 'ndvi_raw_appEARS_combined.csv'
NDVI_QC_PATH = EXTERNAL_DIR / 'ndvi_jja_bitmask_qc_filtered.csv'
NDVI_DUP_PATH = EXTERNAL_DIR / 'ndvi_duplicate_observation_keys.csv'
EXTERNAL_PANEL_PATH = EXTERNAL_DIR / 'external_features.csv'
TASK_IDS_PATH = EXTERNAL_DIR / 'appeears_task_ids.json'
NDVI_METADATA_PATH = EXTERNAL_DIR / 'ndvi_raw_metadata.json'
DEM_METADATA_PATH = EXTERNAL_DIR / 'dem_elevation_metadata.json'

print(f'Run ID: {RUN_ID}; eligible cities: {len(eligible):,}; China={(eligible.Country=="China").sum():,}; Türkiye={(eligible.Country=="Türkiye").sum():,}')
print(f'Project directory: {PROJECT_DIR}')
print(f'Phase 2 root: {PHASE2_ROOT}\nExternal feature root: {EXTERNAL_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archived 17 existing external-feature files to /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/previous_output_backups/external_20261019_142158
Run ID: 20261019_142159; eligible cities: 1,621; China=1,441; Türkiye=180
Project directory: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer
Phase 2 root: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/phase2_outputs
External feature root: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/data/external_features


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path(
    "/content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/"
    "urban-heat-transfer/data/external_features/"
    "ndvi_raw_appEARS_combined.csv"
)

df_ndvi = pd.read_csv(csv_path)

print("Toplam satır:", len(df_ndvi))
print("Benzersiz şehir:", df_ndvi["ID"].nunique())

df_ndvi["Date"] = pd.to_datetime(df_ndvi["Date"], errors="coerce")
print("İlk tarih:", df_ndvi["Date"].min())
print("Son tarih:", df_ndvi["Date"].max())

print("Tekrarlanan şehir-tarih kayıtları:",
      df_ndvi.duplicated(subset=["ID", "Date"]).sum())

ndvi_col = "MOD13Q1_061__250m_16_days_NDVI"
print("Eksik NDVI değeri:", df_ndvi[ndvi_col].isna().sum())
print("Özel dolgu değeri (-3000) sayısı:",
      (df_ndvi[ndvi_col] == -3000).sum())

Toplam satır: 226940
Benzersiz şehir: 1621
İlk tarih: 2001-05-25 00:00:00
Son tarih: 2020-08-28 00:00:00
Tekrarlanan şehir-tarih kayıtları: 0
Eksik NDVI değeri: 0
Özel dolgu değeri (-3000) sayısı: 1101


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# Prepare the NASA NDVI data for summer analysis.
ndvi_col = "MOD13Q1_061__250m_16_days_NDVI"
quality_col = "MOD13Q1_061__250m_16_days_VI_Quality"

df = df_ndvi[["ID", "Date", ndvi_col, quality_col]].copy()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# Convert fill values to missing values.
df["NDVI"] = pd.to_numeric(df[ndvi_col], errors="coerce")
df.loc[df["NDVI"] == -3000, "NDVI"] = np.nan

# Decode MODIS quality fields.
qa = pd.to_numeric(df[quality_col], errors="coerce")
qa = qa.fillna(-1).astype("int64")
df["QA_flag"] = qa & 0b11
df["VI_usefulness"] = (qa // 4) % 16

# Retain valid NDVI observations from June, July, and August.
valid = (
    df["NDVI"].between(-0.2, 1.0)
    & df["QA_flag"].isin([0, 1])
    & df["VI_usefulness"].le(12)
    & df["Date"].notna()
    & df["Date"].dt.month.isin([6, 7, 8])
)
summer = df.loc[valid].copy()
summer["Year"] = summer["Date"].dt.year

# Calculate summer NDVI features for each city and year.
features = (
    summer.groupby(["ID", "Year"])["NDVI"]
    .agg(
        NDVI_JJA_mean="mean",
        NDVI_JJA_median="median",
        NDVI_JJA_std="std",
        NDVI_JJA_n="count"
    )
    .reset_index()
)

# Include all cities and years, even when observations are unavailable.
city_ids = df_ndvi["ID"].dropna().drop_duplicates().tolist()
years = range(2001, 2021)
full_index = pd.MultiIndex.from_product(
    [city_ids, years], names=["ID", "Year"]
)

features = (
    features.set_index(["ID", "Year"])
    .reindex(full_index)
    .reset_index()
    .rename(columns={"ID": "UrbanId"})
)
features["NDVI_JJA_n"] = features["NDVI_JJA_n"].fillna(0).astype(int)

# Save the processed features separately from the raw data.
project_dir = Path(
    "/content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/"
    "urban-heat-transfer"
)
output_dir = project_dir / "data" / "external_features"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "ndvi_jja_city_year.csv"
features.to_csv(output_path, index=False)

print("NDVI hazırlandı ve kaydedildi:", output_path)
print("Toplam şehir-yıl satırı:", len(features))
print("Gözlem bulunan şehir-yıl sayısı:",
      (features["NDVI_JJA_n"] > 0).sum())

NDVI hazırlandı ve kaydedildi: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/data/external_features/ndvi_jja_city_year.csv
Toplam şehir-yıl satırı: 32420
Gözlem bulunan şehir-yıl sayısı: 31203


## Step 1 — DEM / Elevation

If the old elevation table doesn't contain all the cities in the target manifest, it's backed up and recreated. Because a city row is generated from each API request, an API error won't silently destroy the city record; the elevation will be recorded as NaN and will appear in the quality report.

In [ ]:
# Open-Meteo Elevation API uses Copernicus DEM GLO-90 (90 m); preserve source and coordinate provenance.
DEM_SOURCE = 'Open-Meteo Elevation API / Copernicus DEM GLO-90'
expected_ids = set(eligible['UrbanId'].astype(int))
coordinate_frame = eligible[['UrbanId','Latitude','Longitude']].copy().sort_values('UrbanId')
coordinate_signature = hashlib.sha256(coordinate_frame.to_csv(index=False, float_format='%.8f').encode('utf-8')).hexdigest()

def valid_dem_file(path):
    try:
        if not path.exists() or not DEM_METADATA_PATH.exists(): return False
        meta = json.loads(DEM_METADATA_PATH.read_text(encoding='utf-8'))
        if meta.get('source') != DEM_SOURCE or meta.get('coordinate_signature') != coordinate_signature:
            return False
        if meta.get('file_sha256') != file_sha256(path):
            return False
        d = pd.read_csv(path)
        if not {'UrbanId','elevation_m','dem_source'}.issubset(d.columns): return False
        d['UrbanId'] = pd.to_numeric(d['UrbanId'], errors='coerce')
        d['elevation_m'] = pd.to_numeric(d['elevation_m'], errors='coerce')
        ids = set(d['UrbanId'].dropna().astype(int))
        return (not d['UrbanId'].isna().any() and not d['UrbanId'].duplicated().any()
                and ids == expected_ids and d['dem_source'].eq(DEM_SOURCE).all())
    except Exception:
        return False

if valid_dem_file(DEM_PATH):
    dem = pd.read_csv(DEM_PATH)
    print(f'Reusing provenance-validated DEM table: {DEM_PATH}, rows={len(dem):,}')
else:
    if DEM_PATH.exists():
        backup = DEM_PATH.with_name(f'dem_elevation_unverified_backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv')
        shutil.copy2(DEM_PATH, backup)
        print(f'Preserved existing DEM table without matching coordinate/source provenance: {backup}')
    records = []
    coords = eligible[['UrbanId','Latitude','Longitude']].reset_index(drop=True)
    batch_size = 100  # Open-Meteo documents up to 100 coordinate pairs per request.
    for start in range(0, len(coords), batch_size):
        batch = coords.iloc[start:start+batch_size]
        params = {
            'latitude': ','.join(f'{x:.6f}' for x in batch['Latitude']),
            'longitude': ','.join(f'{x:.6f}' for x in batch['Longitude']),
        }
        elevations = None
        for attempt in range(3):
            try:
                response = requests.get('https://api.open-meteo.com/v1/elevation', params=params, timeout=45)
                response.raise_for_status()
                elevations = response.json().get('elevation', [])
                if not isinstance(elevations, list): raise ValueError('Elevation API response field is not a list.')
                break
            except Exception as exc:
                if attempt == 2:
                    print(f'API request failed for batch {start}:{start+len(batch)}: {type(exc).__name__}: {exc}')
                else: time.sleep(2 ** attempt)
        elevations = elevations or []
        if len(elevations) != len(batch):
            print(f'WARNING: DEM batch {start//batch_size+1} returned {len(elevations)} values for {len(batch)} coordinates; unmatched values will remain missing.')
        for j, row in enumerate(batch.itertuples(index=False)):
            value = elevations[j] if j < len(elevations) else np.nan
            records.append({'UrbanId':int(row.UrbanId),'elevation_m':pd.to_numeric(value, errors='coerce'),
                            'dem_source':DEM_SOURCE,'coordinate_signature':coordinate_signature})
        time.sleep(0.3)
    dem = pd.DataFrame(records)
    if dem['UrbanId'].duplicated().any(): raise AssertionError('DEM collector emitted duplicate UrbanId rows.')
    dem.to_csv(DEM_PATH, index=False)
    DEM_METADATA_PATH.write_text(json.dumps({
        'status':'collected','source':DEM_SOURCE,'coordinate_signature':coordinate_signature,
        'requested_cities':len(expected_ids),'returned_rows':int(len(dem)),
        'missing_elevation_rows':int(dem['elevation_m'].isna().sum()),
        'generated_at':datetime.now().isoformat()
    }, indent=2), encoding='utf-8')
    print(f'Wrote {DEM_PATH}; rows={len(dem):,}; missing elevations={int(dem.elevation_m.isna().sum())}')

# Canonical one-row-per-eligible-city table. Missing elevations are preserved and reported, not fabricated.
dem = eligible[['UrbanId']].merge(dem[['UrbanId','elevation_m','dem_source']], on='UrbanId', how='left', validate='one_to_one')
dem['dem_source'] = dem['dem_source'].fillna(DEM_SOURCE)
dem.to_csv(DEM_PATH, index=False)
# Refresh provenance against the exact canonical bytes just written (also when reusing a cache).
DEM_METADATA_PATH.write_text(json.dumps({
    'status':'canonical_table_validated','source':DEM_SOURCE,
    'coordinate_signature':coordinate_signature,'requested_cities':len(expected_ids),
    'returned_rows':int(len(dem)),'missing_elevation_rows':int(dem['elevation_m'].isna().sum()),
    'file_sha256':file_sha256(DEM_PATH),'file_bytes':DEM_PATH.stat().st_size,
    'generated_at':datetime.now().isoformat()
},indent=2),encoding='utf-8')
dem_missing = dem[dem['elevation_m'].isna()][['UrbanId']].copy()
dem_missing['reason'] = 'Elevation API did not return a usable value'
dem_missing.to_csv(EXTERNAL_DIR / 'audit_dem_missing_cities.csv', index=False)
print(f'DEM coverage: {dem.elevation_m.notna().sum():,}/{len(dem):,}; missing={dem.elevation_m.isna().sum():,}')

API request failed for batch 600:700: HTTPError: 429 Client Error: Too Many Requests for url: https://api.open-meteo.com/v1/elevation?latitude=37.506252%2C35.390836%2C35.509169%2C37.754863%2C39.746961%2C22.991266%2C39.809124%2C33.447306%2C30.991795%2C31.170078%2C28.065888%2C38.596298%2C36.510359%2C38.691206%2C28.401044%2C36.603216%2C36.640359%2C31.721341%2C34.529659%2C31.894573%2C37.359883%2C21.609169%2C41.386601%2C29.719794%2C26.262503%2C39.692595%2C30.000002%2C39.872456%2C35.570100%2C30.241669%2C33.304927%2C30.657554%2C27.351750%2C23.351114%2C30.948961%2C23.894447%2C24.666669%2C36.911032%2C34.550381%2C25.103614%2C41.253243%2C34.910229%2C32.262763%2C25.453336%2C32.392971%2C32.542971%2C39.723216%2C37.732600%2C29.707664%2C37.819855%2C26.222780%2C37.913728%2C21.915664%2C26.602780%2C30.232664%2C22.791813%2C22.999859%2C36.326896%2C33.777867%2C31.026481%2C36.486745%2C31.262771%2C40.935359%2C23.943250%2C24.123422%2C39.102943%2C28.296391%2C47.723613%2C39.418629%2C24.788077%2C34.919794%2C37.42

NameError: name 'file_sha256' is not defined

In [ ]:
# Repair missing DEM values without downloading the entire dataset again.

import hashlib
import json
import time
import requests
import numpy as np
import pandas as pd
from datetime import datetime
from pathlib import Path

DEM_SOURCE = "Open-Meteo Elevation API / Copernicus DEM GLO-90"

# Keep the same coordinate signature used by the original notebook.
coordinate_frame = (
    eligible[["UrbanId", "Latitude", "Longitude"]]
    .copy()
    .sort_values("UrbanId")
)

coordinate_signature = hashlib.sha256(
    coordinate_frame.to_csv(
        index=False, float_format="%.8f"
    ).encode("utf-8")
).hexdigest()

def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

dem_path = Path(DEM_PATH)
metadata_path = Path(DEM_METADATA_PATH)

coords = coordinate_frame[["UrbanId", "Latitude", "Longitude"]].copy()
coords["UrbanId"] = pd.to_numeric(
    coords["UrbanId"], errors="raise"
).astype(int)

# Load and preserve any elevations already downloaded.
if dem_path.exists():
    old_dem = pd.read_csv(dem_path)

    if (
        "elevation_m" not in old_dem.columns
        and "elevation_srtm" in old_dem.columns
    ):
        old_dem = old_dem.rename(
            columns={"elevation_srtm": "elevation_m"}
        )

    if not {"UrbanId", "elevation_m"}.issubset(old_dem.columns):
        raise ValueError("Unexpected columns in the existing DEM file.")

    old_dem["UrbanId"] = pd.to_numeric(
        old_dem["UrbanId"], errors="coerce"
    )
    old_dem["elevation_m"] = pd.to_numeric(
        old_dem["elevation_m"], errors="coerce"
    )
    old_dem = old_dem.dropna(subset=["UrbanId"]).copy()
    old_dem["UrbanId"] = old_dem["UrbanId"].astype(int)
    old_dem = old_dem.drop_duplicates("UrbanId", keep="last")

else:
    old_dem = pd.DataFrame(columns=["UrbanId", "elevation_m"])

dem = coords[["UrbanId"]].merge(
    old_dem[["UrbanId", "elevation_m"]],
    on="UrbanId",
    how="left",
    validate="one_to_one",
)

dem["dem_source"] = DEM_SOURCE

# Save a valid, resumable checkpoint after every successful batch.
def save_dem_checkpoint():
    dem["dem_source"] = DEM_SOURCE
    dem.to_csv(dem_path, index=False)

    metadata = {
        "status": "repair_in_progress",
        "source": DEM_SOURCE,
        "coordinate_signature": coordinate_signature,
        "requested_cities": len(coords),
        "returned_rows": len(dem),
        "missing_elevation_rows": int(dem["elevation_m"].isna().sum()),
        "file_sha256": file_sha256(dem_path),
        "file_bytes": dem_path.stat().st_size,
        "generated_at": datetime.now().isoformat(),
    }

    metadata_path.write_text(
        json.dumps(metadata, indent=2),
        encoding="utf-8",
    )

# Record a checkpoint before requesting more data.
save_dem_checkpoint()

missing_ids = set(
    dem.loc[dem["elevation_m"].isna(), "UrbanId"].astype(int)
)
missing_coords = coords[
    coords["UrbanId"].isin(missing_ids)
].copy()

print("Total cities:", len(coords))
print("Cities still missing elevation:", len(missing_coords))
print("Existing valid elevations:", int(dem["elevation_m"].notna().sum()))

session = requests.Session()
batch_size = 10

for start in range(0, len(missing_coords), batch_size):
    batch = missing_coords.iloc[start:start + batch_size]
    success = False
    last_error = None

    for attempt in range(6):
        try:
            response = session.get(
                "https://api.open-meteo.com/v1/elevation",
                params={
                    "latitude": ",".join(
                        f"{x:.6f}" for x in batch["Latitude"]
                    ),
                    "longitude": ",".join(
                        f"{x:.6f}" for x in batch["Longitude"]
                    ),
                },
                timeout=45,
            )

            if response.status_code == 429:
                wait = min(60, 5 * (2 ** attempt))
                print(f"Rate limit reached; waiting {wait} seconds.")
                time.sleep(wait)
                continue

            response.raise_for_status()
            values = response.json().get("elevation")

            if not isinstance(values, list) or len(values) != len(batch):
                raise ValueError("Unexpected number of elevation values.")

            updates = dict(
                zip(
                    batch["UrbanId"].astype(int),
                    pd.to_numeric(
                        pd.Series(values), errors="coerce"
                    ).to_numpy(),
                )
            )

            mask = dem["UrbanId"].isin(updates)
            dem.loc[mask, "elevation_m"] = (
                dem.loc[mask, "UrbanId"].map(updates).to_numpy()
            )

            save_dem_checkpoint()

            print(
                f"Processed {min(start + len(batch), len(missing_coords))}"
                f"/{len(missing_coords)} missing cities."
            )

            success = True
            break

        except Exception as exc:
            last_error = str(exc)

            if attempt < 5:
                wait = min(60, 5 * (2 ** attempt))
                print(f"Request failed; retrying in {wait}s.")
                time.sleep(wait)

    if not success:
        save_dem_checkpoint()
        raise RuntimeError(
            f"Could not complete batch starting at row {start}. "
            f"Progress saved. Last error: {last_error}"
        )

    time.sleep(2)

# Final report and missing-city audit.
save_dem_checkpoint()

missing = dem[dem["elevation_m"].isna()][["UrbanId"]].copy()
missing["reason"] = "Elevation API did not return a usable value"

missing.to_csv(
    Path(EXTERNAL_DIR) / "audit_dem_missing_cities.csv",
    index=False,
)

metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
metadata["status"] = (
    "canonical_table_validated"
    if missing.empty
    else "repair_incomplete"
)
metadata["generated_at"] = datetime.now().isoformat()

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("DEM coverage:", f"{dem['elevation_m'].notna().sum()}/{len(dem)}")
print("Remaining missing elevations:", len(missing))
print("Saved DEM:", dem_path)

Total cities: 1621
Cities still missing elevation: 411
Existing valid elevations: 1210
Processed 10/411 missing cities.
Processed 20/411 missing cities.
Processed 30/411 missing cities.
Processed 40/411 missing cities.
Processed 50/411 missing cities.
Processed 60/411 missing cities.
Processed 70/411 missing cities.
Processed 80/411 missing cities.
Processed 90/411 missing cities.
Processed 100/411 missing cities.
Processed 110/411 missing cities.
Processed 120/411 missing cities.
Processed 130/411 missing cities.
Processed 140/411 missing cities.
Processed 150/411 missing cities.
Processed 160/411 missing cities.
Processed 170/411 missing cities.
Processed 180/411 missing cities.
Processed 190/411 missing cities.
Processed 200/411 missing cities.
Processed 210/411 missing cities.
Processed 220/411 missing cities.
Processed 230/411 missing cities.
Processed 240/411 missing cities.
Processed 250/411 missing cities.
Processed 260/411 missing cities.
Processed 270/411 missing cities.
Proc

## Step 2 — ERA5-Land Monthly Data Download and JJA Features

Data used: ERA5-Land monthly means, June/July/August 2001–2020. The CDS API private key is not written to the notebook; it is read from the Colab Secret `CDS_API_KEY` and written only to the runtime `~/.cdsapirc` file. If an existing NetCDF exists, the content and date range are checked; it is not accepted as correct based solely on the file name.

In [ ]:
import subprocess, sys
# Notebook-level install. Package versions used by this run are recorded in the QA report.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cdsapi>=0.7.7', 'netCDF4', 'xarray'], check=True)
import cdsapi

ERA5_REQUEST = {
    'dataset': 'reanalysis-era5-land-monthly-means',
    'product_type': 'monthly_averaged_reanalysis',
    'variable': ['2m_temperature','total_precipitation','10m_u_component_of_wind',
                 '10m_v_component_of_wind','2m_dewpoint_temperature'],
    'year': [str(y) for y in YEARS],
    'month': ['06','07','08'],
    'time': ['00:00'],
    'area': [55,25,18,135],
    'data_format': 'netcdf',
    'download_format': 'unarchived',
}
ERA5_REQUEST_FINGERPRINT = hashlib.sha256(json.dumps(ERA5_REQUEST,sort_keys=True).encode('utf-8')).hexdigest()

def file_sha256(path, chunk_size=8*1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(chunk_size),b''):
            h.update(block)
    return h.hexdigest()

def validate_era5_structure(path):
    """Check product-like structure and units, but do not certify provenance by structure alone."""
    try:
        ds = xr.open_dataset(path)
        vars_ok = {'t2m','tp','u10','v10','d2m'}.issubset(set(ds.data_vars))
        time_name = next((c for c in ds.coords if 'time' in c.lower()), None)
        lat_name = 'latitude' if 'latitude' in ds.coords else ('lat' if 'lat' in ds.coords else None)
        lon_name = 'longitude' if 'longitude' in ds.coords else ('lon' if 'lon' in ds.coords else None)
        if not vars_ok or time_name is None or lat_name is None or lon_name is None:
            ds.close(); return False, 'required variables/time/lat/lon missing'
        vals = pd.to_datetime(ds[time_name].values)
        month_pairs = [(t.year,t.month) for t in vals]
        expected = {(y,m) for y in YEARS for m in [6,7,8]}
        month_counts = pd.Series(month_pairs).value_counts().to_dict()
        unique_complete = expected.issubset(set(month_pairs)) and all(month_counts.get(k,0)==1 for k in expected)
        # ECMWF documents that total precipitation in ERA5-Land monthly means of daily means
        # is expressed in m water-equivalent per day. Some NetCDF exports expose only 'm' in
        # the units attribute; that ambiguity is resolved only by a matching sidecar that records
        # this exact CDS product request, never by guessing from a legacy filename alone.
        tp_units = str(ds['tp'].attrs.get('units','')).strip().lower().replace(' ','')
        t2m_units = str(ds['t2m'].attrs.get('units','')).strip().lower()
        d2m_units = str(ds['d2m'].attrs.get('units','')).strip().lower()
        u10_units = str(ds['u10'].attrs.get('units','')).lower().replace(' ','')
        v10_units = str(ds['v10'].attrs.get('units','')).lower().replace(' ','')
        tp_units_plausible = (('day' in tp_units) or ('d-1' in tp_units) or ('day**-1' in tp_units)
                              or tp_units in {'m','meter','metre','meters','metres'})
        units_ok = tp_units_plausible and t2m_units in {'k','kelvin'} and d2m_units in {'k','kelvin'}
        units_ok = units_ok and all(('m/s' in u or 'ms-1' in u or 'ms**-1' in u or 'm.s-1' in u) for u in [u10_units,v10_units])
        latvals = np.asarray(ds[lat_name].values,dtype=float)
        lonvals = np.asarray(ds[lon_name].values,dtype=float)
        extent_ok = (latvals.min() <= float(eligible['Latitude'].min()) and latvals.max() >= float(eligible['Latitude'].max())
                     and lonvals.min() <= float(eligible['Longitude'].min()) and lonvals.max() >= float(eligible['Longitude'].max()))
        details = f'vars={vars_ok}; months={unique_complete}; units={units_ok}; extent={extent_ok}; tp_units={tp_units!r}'
        okay = vars_ok and unique_complete and units_ok and extent_ok
        ds.close()
        return okay, details
    except Exception as exc:
        return False, f'{type(exc).__name__}: {exc}'

def is_valid_era5(path):
    """A structurally plausible legacy file is not reused without matching request+SHA provenance."""
    try:
        path = Path(path)
        ok, _ = validate_era5_structure(path)
        if not ok: return False
        sidecar = path.with_name(path.stem + '_metadata.json')
        if not sidecar.exists(): return False
        meta = json.loads(sidecar.read_text(encoding='utf-8'))
        return (meta.get('request_fingerprint') == ERA5_REQUEST_FINGERPRINT
                and meta.get('file_sha256') == file_sha256(path)
                and meta.get('source_product') == ERA5_REQUEST['dataset'])
    except Exception:
        return False

legacy_nc_candidates = [
    ERA5_NC_PATH,
    PROJECT_DIR / 'phase2_output/era5_land_jja_2001_2020.nc',
    PROJECT_DIR / 'phase2_outputtt/era5_land_jja_2001_2020.nc',
    PROJECT_DIR / 'phase2_output/era5_land_jja_2001_2020(1).nc',
    PROJECT_DIR / 'phase2_outputtt/era5_land_jja_2001_2020(1).nc',
    DRIVE_ROOT / 'phase2_output/era5_land_jja_2001_2020.nc',
    DRIVE_ROOT / 'phase2_outputtt/era5_land_jja_2001_2020.nc',
    DRIVE_ROOT / 'phase2_output/era5_land_jja_2001_2020(1).nc',
    DRIVE_ROOT / 'phase2_outputtt/era5_land_jja_2001_2020(1).nc',
]
valid_candidate = next((p for p in legacy_nc_candidates if p.exists() and is_valid_era5(p)), None)
if valid_candidate is not None:
    if valid_candidate != ERA5_NC_PATH:
        shutil.copy2(valid_candidate, ERA5_NC_PATH)
        shutil.copy2(valid_candidate.with_name(valid_candidate.stem+'_metadata.json'), ERA5_METADATA_PATH)
        print(f'Copied provenance-validated legacy NetCDF to canonical path: {ERA5_NC_PATH}')
    else:
        print(f'Using provenance-validated ERA5 NetCDF: {ERA5_NC_PATH}')
else:
    print('No NetCDF with matching current request fingerprint and file hash found. A fresh CDS download will be required; old files will be preserved.')


No NetCDF with matching current request fingerprint and file hash found. A fresh CDS download will be required; old files will be preserved.


In [ ]:
if not is_valid_era5(ERA5_NC_PATH):
    cds_key = userdata.get('CDS_API_KEY')
    if not cds_key:
        raise RuntimeError('Add CDS_API_KEY to Google Colab > Secrets and enable notebook access. Do not paste credentials into cells.')

    # Preserve any legacy/unverified file before replacing the canonical path.
    if ERA5_NC_PATH.exists():
        backup_dir = BACKUP_ROOT / f'era5_unverified_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
        backup_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(ERA5_NC_PATH, backup_dir / ERA5_NC_PATH.name)
        old_sidecar = ERA5_NC_PATH.with_name(ERA5_NC_PATH.stem+'_metadata.json')
        if old_sidecar.exists(): shutil.copy2(old_sidecar, backup_dir / old_sidecar.name)
        print(f'Preserved unverified ERA5 file before fresh download: {backup_dir}')

    cds_url = 'https://cds.climate.copernicus.eu/api'
    cdsrc = Path.home() / '.cdsapirc'
    cdsrc.write_text(f'url: {cds_url}\nkey: {cds_key}\n', encoding='utf-8')
    try: os.chmod(cdsrc, 0o600)
    except Exception: pass

    tmp_path = ERA5_NC_PATH.with_suffix('.download')
    if tmp_path.exists(): tmp_path.unlink()
    try:
        client = cdsapi.Client()  # cdsapi reads ~/.cdsapirc during construction
    finally:
        # Avoid leaving the API key in a persistent Colab home file after the client reads it.
        try:
            cdsrc.unlink()
        except FileNotFoundError:
            pass
    request = {k:v for k,v in ERA5_REQUEST.items() if k!='dataset'}
    client.retrieve(ERA5_REQUEST['dataset'], request, str(tmp_path))
    if zipfile.is_zipfile(tmp_path):
        with zipfile.ZipFile(tmp_path) as zf:
            nc_members = [n for n in zf.namelist() if n.lower().endswith('.nc')]
            if len(nc_members) != 1:
                raise RuntimeError(f'Expected exactly one NetCDF in CDS archive, found {nc_members[:10]}')
            extracted = PHASE2_ROOT / f'era5_download_extracted_{RUN_ID}.nc'
            with zf.open(nc_members[0]) as src, open(extracted,'wb') as dst:
                shutil.copyfileobj(src,dst)
            shutil.move(extracted, ERA5_NC_PATH)
    else:
        shutil.move(tmp_path, ERA5_NC_PATH)
    if tmp_path.exists(): tmp_path.unlink()

    structure_ok, structure_details = validate_era5_structure(ERA5_NC_PATH)
    if not structure_ok:
        raise RuntimeError(f'Downloaded ERA5-Land NetCDF failed structural/units/coverage checks: {structure_details}. Preserved older file remains in backup.')
    meta = {
        'status':'downloaded_and_validated', 'source_product':ERA5_REQUEST['dataset'],
        'request_fingerprint':ERA5_REQUEST_FINGERPRINT, 'request':ERA5_REQUEST,
        'file_sha256':file_sha256(ERA5_NC_PATH), 'file_bytes':ERA5_NC_PATH.stat().st_size,
        'structure_validation':structure_details, 'generated_at':datetime.now().isoformat(),
    }
    ERA5_METADATA_PATH.write_text(json.dumps(meta,indent=2),encoding='utf-8')
    if not is_valid_era5(ERA5_NC_PATH):
        raise RuntimeError('Downloaded ERA5 file failed provenance + hash validation after sidecar creation.')
    print(f'Fresh ERA5-Land data saved and provenance-validated: {ERA5_NC_PATH} ({ERA5_NC_PATH.stat().st_size/1024/1024:.1f} MB)')
else:
    print('ERA5 NetCDF has matching request provenance and file hash; no re-download necessary.')


2026-10-18 12:32:43,161 INFO [2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 October 2026.
 **[More information about the upcoming change.](https://forum.ecmwf.int/t/upcoming-cds-terms-of-use-change/15374)**
INFO:ecmwf.datastores.legacy_client:[2026-10-06T00:00:00Z] The [CDS Terms of Use](https://cds.climate.copernicus.eu/disclaimer-privacy) will be updated on 28 October 2026. Continued use of Copernicus Products and Services through the CDS after this date constitutes acceptance of the updated terms. If you do not accept the change, please [contact ECMWF Support](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) before 27 

b1c7860a2ab65dace32e9f7be274f026.nc:   0%|          | 0.00/197M [00:00<?, ?B/s]

Fresh ERA5-Land data saved and provenance-validated: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/phase2_outputs/era5_land_jja_2001_2020.nc (196.5 MB)


In [12]:
# FAST ERA5-LAND EXTRACTION — vectorized version
# Uses the existing NetCDF and spatial assignments.
# Doesnot download source data again.


import json
import hashlib
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr


# 1. Paths and configuration
PROJECT_DIR = Path(
    "/content/drive/MyDrive/"
    "Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer"
)
PHASE2_ROOT = PROJECT_DIR / "outputs" / "phase2_outputs"
EXTERNAL_DIR = PROJECT_DIR / "data" / "external_features"

NC_PATH = PHASE2_ROOT / "era5_land_jja_2001_2020.nc"
META_PATH = PHASE2_ROOT / "era5_land_jja_2001_2020_metadata.json"
CSV_PATH = PHASE2_ROOT / "era5_jja_features.csv"
AUDIT_PATH = EXTERNAL_DIR / "era5_grid_cell_assignment_audit.csv"

YEARS = list(range(2001, 2021))
MONTHS = [6, 7, 8]
VARIABLES = ["t2m", "tp", "u10", "v10", "d2m"]
MAX_FALLBACK_KM = 20.0

# The interrupted cell should leave these two objects in memory.
if "eligible" not in globals():
    raise RuntimeError(
        "`eligible` is not in memory. Run setup cells only; "
        "do not restart the runtime or repeat downloads."
    )

if "assignment_df" not in globals():
    raise RuntimeError(
        "`assignment_df` is not in memory. The spatial assignment "
        "stage must finish successfully before this fast extraction."
    )

cities = eligible[
    ["UrbanId", "Country", "Latitude", "Longitude"]
].copy()

cities["UrbanId"] = pd.to_numeric(
    cities["UrbanId"], errors="raise"
).astype(int)

assignments = (
    assignment_df.copy()
    .sort_values("UrbanId")
    .reset_index(drop=True)
)

cities = cities.sort_values("UrbanId").reset_index(drop=True)

if len(cities) != 1621 or cities["UrbanId"].duplicated().any():
    raise ValueError("Eligible city cohort failed validation.")

if len(assignments) != len(cities):
    raise ValueError("Spatial assignment count does not match cities.")

if assignments["UrbanId"].duplicated().any():
    raise ValueError("Duplicate city IDs exist in spatial assignments.")

if set(assignments["UrbanId"]) != set(cities["UrbanId"]):
    raise ValueError("Spatial assignments do not match eligible cities.")

if assignments["distance_km"].max() > MAX_FALLBACK_KM:
    raise ValueError("A selected grid cell exceeds the 20 km limit.")

print("Eligible cities:", len(cities))
print("Spatial fallback cities:", int(assignments["fallback_used"].sum()))
print("Preparing vectorized extraction...")


# 2. Verify the saved source and request provenance
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


if not NC_PATH.is_file() or not META_PATH.is_file():
    raise FileNotFoundError("Existing ERA5 NetCDF or metadata is missing.")

metadata = json.loads(META_PATH.read_text(encoding="utf-8"))
source_hash = file_sha256(NC_PATH)

if metadata.get("file_sha256") != source_hash:
    raise ValueError("ERA5 source hash does not match its metadata.")

ERA5_REQUEST = {
    "dataset": "reanalysis-era5-land-monthly-means",
    "product_type": "monthly_averaged_reanalysis",
    "variable": [
        "2m_temperature",
        "total_precipitation",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "2m_dewpoint_temperature",
    ],
    "year": [str(y) for y in YEARS],
    "month": ["06", "07", "08"],
    "time": ["00:00"],
    "area": [55, 25, 18, 135],
    "data_format": "netcdf",
    "download_format": "unarchived",
}

expected_fingerprint = hashlib.sha256(
    json.dumps(ERA5_REQUEST, sort_keys=True).encode("utf-8")
).hexdigest()

if metadata.get("source_product") != "reanalysis-era5-land-monthly-means":
    raise ValueError("Unexpected ERA5 source product.")

if metadata.get("request_fingerprint") != expected_fingerprint:
    raise ValueError("ERA5 request fingerprint does not match.")

print("Source hash and request fingerprint verified.")


# 3. Read all selected-city grid values in vectorized batches
with xr.open_dataset(NC_PATH) as ds:

    time_name = next(
        (c for c in ds.coords if "time" in c.lower()), None
    )
    lat_name = (
        "latitude" if "latitude" in ds.coords
        else "lat" if "lat" in ds.coords else None
    )
    lon_name = (
        "longitude" if "longitude" in ds.coords
        else "lon" if "lon" in ds.coords else None
    )

    if not all([time_name, lat_name, lon_name]):
        raise ValueError("Could not identify ERA5 coordinate axes.")

    if not set(VARIABLES).issubset(ds.data_vars):
        raise ValueError("Required ERA5 variables are missing.")

    times = pd.DatetimeIndex(pd.to_datetime(ds[time_name].values))

    month_index = {}
    for i, timestamp in enumerate(times):
        month_index.setdefault(
            (timestamp.year, timestamp.month), []
        ).append(i)

    expected_months = {
        (y, m) for y in YEARS for m in MONTHS
    }

    missing_months = expected_months - set(month_index)
    duplicate_months = [
        key for key in expected_months
        if len(month_index.get(key, [])) != 1
    ]

    if missing_months or duplicate_months:
        raise ValueError(
            f"Time-axis validation failed. "
            f"Missing={sorted(missing_months)}, "
            f"duplicate={duplicate_months}"
        )

    time_indices = [
        month_index[(y, m)][0]
        for y in YEARS for m in MONTHS
    ]

    lats = np.asarray(ds[lat_name].values, dtype=float)
    lons = np.asarray(ds[lon_name].values, dtype=float)

    # Convert the selected geographic coordinates into grid indices.
    city_lat = assignments["selected_grid_latitude"].to_numpy(float)
    city_lon = assignments["selected_grid_longitude"].to_numpy(float)

    lat_indices = np.array([
        int(np.abs(lats - lat).argmin()) for lat in city_lat
    ])

    lon_indices = np.array([
        int(np.abs((lons - lon + 180) % 360 - 180).argmin())
        for lon in city_lon
    ])

    # Paired indexers select one cell per city rather than every
    # latitude-longitude combination.
    city_lat_indexer = xr.DataArray(
        lat_indices, dims="city"
    )
    city_lon_indexer = xr.DataArray(
        lon_indices, dims="city"
    )

    selected_arrays = {}

    for variable in VARIABLES:
        da = ds[variable]

        # Remove unexpected dimensions only when they are singleton.
        for dim in list(da.dims):
            if dim not in {time_name, lat_name, lon_name}:
                if da.sizes[dim] != 1:
                    raise ValueError(
                        f"Unexpected non-singleton dimension {dim} "
                        f"in {variable}."
                    )
                da = da.isel({dim: 0})

        if not {time_name, lat_name, lon_name}.issubset(da.dims):
            raise ValueError(
                f"Required coordinates are missing from {variable}."
            )

        selected = da.isel({
            time_name: time_indices,
            lat_name: city_lat_indexer,
            lon_name: city_lon_indexer,
        }).transpose(time_name, "city")

        values = np.asarray(selected.values, dtype=float)

        if values.shape != (60, len(cities)):
            raise ValueError(
                f"Unexpected selected array shape for {variable}: "
                f"{values.shape}"
            )

        if not np.isfinite(values).all():
            raise ValueError(
                f"Missing/non-finite values remain in selected "
                f"ERA5 variable {variable}."
            )

        # Shape: (20 years, 3 summer months, 1621 cities).
        selected_arrays[variable] = values.reshape(
            len(YEARS), len(MONTHS), len(cities)
        )

        print(f"Loaded {variable}: {values.shape}")

    # Confirm units before calculation.
    t2m_units = str(ds["t2m"].attrs.get("units", "")).lower().strip()
    d2m_units = str(ds["d2m"].attrs.get("units", "")).lower().strip()
    tp_units = str(ds["tp"].attrs.get("units", "")).lower().strip()
    tp_units_compact = tp_units.replace(" ", "")

    u_units = str(ds["u10"].attrs.get("units", "")).lower().replace(" ", "")
    v_units = str(ds["v10"].attrs.get("units", "")).lower().replace(" ", "")

    if t2m_units not in {"k", "kelvin"}:
        raise ValueError(f"Unexpected t2m units: {t2m_units}")

    if d2m_units not in {"k", "kelvin"}:
        raise ValueError(f"Unexpected d2m units: {d2m_units}")

    if not (
        "m/s" in u_units or "ms-1" in u_units
        or "ms**-1" in u_units or "m.s-1" in u_units
    ):
        raise ValueError(f"Unexpected u10 units: {u_units}")

    if not (
        "m/s" in v_units or "ms-1" in v_units
        or "ms**-1" in v_units or "m.s-1" in v_units
    ):
        raise ValueError(f"Unexpected v10 units: {v_units}")

    tp_verified = (
        tp_units_compact in {"m", "meter", "metre", "meters", "metres"}
        and metadata.get("file_sha256") == source_hash
        and metadata.get("request_fingerprint") == expected_fingerprint
    ) or (
        "day" in tp_units_compact
        or "d-1" in tp_units_compact
        or "day**-1" in tp_units_compact
    )

    if not tp_verified:
        raise ValueError(f"Could not verify precipitation units: {tp_units}")


# 4. Calculate the seasonal features using NumPy arrays
t2c = selected_arrays["t2m"] - 273.15
d2c = selected_arrays["d2m"] - 273.15

u10 = selected_arrays["u10"]
v10 = selected_arrays["v10"]
tp = selected_arrays["tp"]

wind = np.sqrt(u10**2 + v10**2)

denominator_t = 243.04 + t2c
denominator_d = 243.04 + d2c

if (
    np.any(np.abs(denominator_t) < 1e-8)
    or np.any(np.abs(denominator_d) < 1e-8)
):
    raise ValueError("Invalid temperature denominator in humidity formula.")

rh = 100.0 * np.exp(
    (17.625 * d2c) / denominator_d
    - (17.625 * t2c) / denominator_t
)
rh = np.clip(rh, 0.0, 100.0)

day_weights = np.array([
    [
        pd.Period(f"{year}-{month:02d}").days_in_month
        for month in MONTHS
    ]
    for year in YEARS
], dtype=float)

weights = day_weights[:, :, None]

def seasonal_weighted_mean(array):
    return np.sum(array * weights, axis=1) / weights.sum(axis=1)

t2m_jja = seasonal_weighted_mean(t2c)
wind_jja = seasonal_weighted_mean(wind)
dewpoint_jja = seasonal_weighted_mean(d2c)
rh_jja = seasonal_weighted_mean(rh)

# Assumes the source's monthly averaged precipitation is mean daily
# accumulation in metres/day, as in the validated original workflow.
precip_jja_mm = np.sum(
    tp * 1000.0 * weights,
    axis=1,
)

print("All city-month values loaded; seasonal calculations completed.")


# 5. Build the output table in the same city/year structure
records = []

for city_index, assignment in enumerate(
    assignments.itertuples(index=False)
):
    uid = int(assignment.UrbanId)

    for year_index, year in enumerate(YEARS):
        records.append({
            "UrbanId": uid,
            "year": year,
            "era5_t2m_jja": float(t2m_jja[year_index, city_index]),
            "era5_precip_jja_mm": float(
                precip_jja_mm[year_index, city_index]
            ),
            "era5_wind_jja": float(wind_jja[year_index, city_index]),
            "era5_dewpoint_jja": float(
                dewpoint_jja[year_index, city_index]
            ),
            "era5_rh_jja": float(rh_jja[year_index, city_index]),
            "era5_months_available": 3,
            "era5_tp_source_units": tp_units,
        })

era5 = pd.DataFrame(records)

feature_columns = [
    "era5_t2m_jja",
    "era5_precip_jja_mm",
    "era5_wind_jja",
    "era5_dewpoint_jja",
    "era5_rh_jja",
]

expected_rows = len(cities) * len(YEARS)

if len(era5) != expected_rows:
    raise AssertionError(
        f"Expected {expected_rows} rows, found {len(era5)}."
    )

if era5.duplicated(["UrbanId", "year"]).any():
    raise AssertionError("Duplicate city-year keys found.")

if set(era5["UrbanId"]) != set(cities["UrbanId"]):
    raise AssertionError("Output city IDs do not match eligible cities.")

if set(era5["year"]) != set(YEARS):
    raise AssertionError("Output year range is incorrect.")

feature_values = era5[feature_columns].to_numpy(dtype=float)

if not np.isfinite(feature_values).all():
    raise ValueError("Non-finite feature values found; output not replaced.")

if (era5["era5_precip_jja_mm"] < 0).any():
    raise ValueError("Negative precipitation found; output not replaced.")

if not era5["era5_rh_jja"].between(0, 100).all():
    raise ValueError("Relative humidity outside 0–100%; output not replaced.")

print("\n=== Extraction QA passed ===")
print("Rows:", len(era5))
print("Cities:", era5["UrbanId"].nunique())
print("Duplicate city-year keys:", era5.duplicated(["UrbanId", "year"]).sum())
print("Missing feature values:", int(era5[feature_columns].isna().sum().sum()))


# 6. Back up existing outputs, then validate and save
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_dir = (
    PROJECT_DIR / "outputs" / "previous_output_backups"
    / f"era5_fast_reextract_{timestamp}"
)

existing = [p for p in [CSV_PATH, AUDIT_PATH] if p.exists()]

if existing:
    backup_dir.mkdir(parents=True, exist_ok=True)
    for path in existing:
        backup_path = backup_dir / path.name
        shutil.copy2(path, backup_path)
        if file_sha256(path) != file_sha256(backup_path):
            raise IOError(f"Backup failed verification: {path.name}")

tmp_csv = CSV_PATH.with_name(f"{CSV_PATH.stem}.tmp_{timestamp}.csv")
tmp_audit = AUDIT_PATH.with_name(f"{AUDIT_PATH.stem}.tmp_{timestamp}.csv")

try:
    era5.to_csv(tmp_csv, index=False)
    assignments.to_csv(tmp_audit, index=False)

    check = pd.read_csv(tmp_csv)
    audit_check = pd.read_csv(tmp_audit)

    if len(check) != expected_rows:
        raise IOError("Saved temporary ERA5 table failed row-count validation.")

    if check.duplicated(["UrbanId", "year"]).any():
        raise IOError("Saved temporary ERA5 table has duplicate keys.")

    if len(audit_check) != len(cities):
        raise IOError("Saved temporary audit has an incorrect row count.")

    tmp_csv.replace(CSV_PATH)
    tmp_audit.replace(AUDIT_PATH)

finally:
    for path in [tmp_csv, tmp_audit]:
        if path.exists():
            path.unlink()

print("\n" + "=" * 55)
print("FAST ERA5 EXTRACTION COMPLETED")
print("=" * 55)
print("ERA5 CSV:", CSV_PATH)
print("Spatial audit:", AUDIT_PATH)
print("Rows:", len(era5))
print("Cities:", era5["UrbanId"].nunique())
print("Years:", era5["year"].min(), "-", era5["year"].max())
print("Fallback cities:", int(assignments["fallback_used"].sum()))
print("Maximum assigned-cell distance (km):",
      round(float(assignments["distance_km"].max()), 3))
print("Missing values by feature:")
print(era5[feature_columns].isna().sum().to_string())

if existing:
    print("Previous outputs backed up to:", backup_dir)

print("\nNext: run the existing Step 4 external-feature merge cell.")

Eligible cities: 1621
Spatial fallback cities: 34
Preparing vectorized extraction...
Source hash and request fingerprint verified.
Loaded t2m: (60, 1621)
Loaded tp: (60, 1621)
Loaded u10: (60, 1621)
Loaded v10: (60, 1621)
Loaded d2m: (60, 1621)
All city-month values loaded; seasonal calculations completed.

=== Extraction QA passed ===
Rows: 32420
Cities: 1621
Duplicate city-year keys: 0
Missing feature values: 0

FAST ERA5 EXTRACTION COMPLETED
ERA5 CSV: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/phase2_outputs/era5_jja_features.csv
Spatial audit: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/data/external_features/era5_grid_cell_assignment_audit.csv
Rows: 32420
Cities: 1621
Years: 2001 - 2020
Fallback cities: 34
Maximum assigned-cell distance (km): 17.129
Missing values by feature:
era5_t2m_jja          0
era5_precip_jja_mm    0
era5_wind_jja         0
era5_dewpoint_jja     0
era5_rh_jja           0
Previous 

## Step 3 — AppEEARS MOD13Q1 NDVI Download and Quality Control

`NASA_EARTHDATA_USERNAME` and `NASA_EARTHDATA_PASSWORD` must be defined in Colab Secrets. If the previously downloaded raw NDVI CSV is available in the correct format, the task will not be reopened; re-downloading the same data is unnecessary. The quality mask will be recalculated with the corrected version.

In [ ]:
APPEEARS_API = 'https://appeears.earthdatacloud.nasa.gov/api/'
NDVI_COL = 'MOD13Q1_061__250m_16_days_NDVI'
QC_COL = 'MOD13Q1_061__250m_16_days_VI_Quality'

# Build a deterministic request fingerprint BEFORE considering any cached/legacy file.
# Old CSVs without a matching metadata sidecar are preserved for audit but are never silently reused.
eligible_points = eligible.sort_values('UrbanId').reset_index(drop=True)
point_rows = eligible_points[['UrbanId','Latitude','Longitude']].copy()
ids_signature = hashlib.sha256(','.join(map(str, point_rows['UrbanId'].astype(int))).encode('utf-8')).hexdigest()
coordinates_signature = hashlib.sha256(point_rows.to_csv(index=False, float_format='%.8f').encode('utf-8')).hexdigest()
chunk_size = 1000
request_signature = {
    'years': YEARS,
    'urban_ids_sha256': ids_signature,
    'coordinates_sha256': coordinates_signature,
    'product': 'MOD13Q1.061',
    'layers': ['_250m_16_days_NDVI','_250m_16_days_VI_Quality'],
    'months_requested': [6,7,8],
    'chunk_size': chunk_size,
    'request_type': 'point',
}
request_fingerprint = hashlib.sha256(json.dumps(request_signature,sort_keys=True).encode('utf-8')).hexdigest()

def ndvi_sidecar(path):
    return path.with_name('ndvi_raw_metadata.json')

def valid_ndvi_raw(path):
    """Only reuse data with the current fingerprint, expected schema and cohort/year coverage."""
    try:
        if not path.exists() or not ndvi_sidecar(path).exists():
            return False
        meta = json.loads(ndvi_sidecar(path).read_text(encoding='utf-8'))
        if meta.get('request_fingerprint') != request_fingerprint:
            return False
        if meta.get('file_sha256') != file_sha256(path):
            return False
        header = pd.read_csv(path, nrows=0)
        if not {'ID','Date',NDVI_COL,QC_COL}.issubset(header.columns):
            return False
        d = pd.read_csv(path, usecols=['ID','Date'])
        d['ID'] = pd.to_numeric(d['ID'], errors='coerce')
        d['Date'] = pd.to_datetime(d['Date'], errors='coerce')
        d = d[d['ID'].notna() & d['Date'].notna()].copy()
        d['ID'] = d['ID'].astype(int)
        d['year'] = d['Date'].dt.year
        d = d[d['year'].isin(YEARS) & d['Date'].dt.month.isin([6,7,8])]
        observed_ids = set(d['ID'].unique())
        observed_years = set(d['year'].astype(int).unique())
        return (expected_ids.issubset(observed_ids) and observed_ids.issubset(expected_ids)
                and set(YEARS).issubset(observed_years) and len(d) > 0)
    except Exception:
        return False

legacy_ndvi_paths = [
    NDVI_RAW_PATH,
    PHASE2_ROOT / 'ndvi_raw_appEARS_combined.csv',
    PROJECT_DIR / 'phase2_output/ndvi_raw_appEARS_combined.csv',
    PROJECT_DIR / 'phase2_outputtt/ndvi_raw_appEARS_combined.csv',
    PROJECT_DIR / 'appeears_output/ndvi_raw_appEARS_combined.csv',
    DRIVE_ROOT / 'phase2_output/ndvi_raw_appEARS_combined.csv',
    DRIVE_ROOT / 'phase2_outputtt/ndvi_raw_appEARS_combined.csv',
    DRIVE_ROOT / 'appeears_output/ndvi_raw_appEARS_combined.csv',
]
legacy_raw = next((p for p in legacy_ndvi_paths if p.exists() and valid_ndvi_raw(p)), None)
if legacy_raw is not None and legacy_raw != NDVI_RAW_PATH:
    NDVI_RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(legacy_raw, NDVI_RAW_PATH)
    shutil.copy2(ndvi_sidecar(legacy_raw), NDVI_METADATA_PATH)
    print(f'Copied provenance-validated raw NDVI to canonical location: {NDVI_RAW_PATH}')

if NDVI_RAW_PATH.exists() and valid_ndvi_raw(NDVI_RAW_PATH):
    print(f'Reusing raw NDVI with matching request fingerprint: {NDVI_RAW_PATH}')
else:
    # Preserve, but do not reuse, files whose coordinates/product/period provenance cannot be verified.
    if NDVI_RAW_PATH.exists():
        backup_dir = EXTERNAL_DIR / 'previous_runs' / f'ndvi_unverified_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
        backup_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(NDVI_RAW_PATH, backup_dir / NDVI_RAW_PATH.name)
        if NDVI_METADATA_PATH.exists(): shutil.copy2(NDVI_METADATA_PATH, backup_dir / NDVI_METADATA_PATH.name)
        print(f'Preserved unverified/stale NDVI files in {backup_dir}; a fresh request will be required.')
        NDVI_RAW_PATH.unlink()
        if NDVI_METADATA_PATH.exists(): NDVI_METADATA_PATH.unlink()

    username = userdata.get('NASA_EARTHDATA_USERNAME')
    password = userdata.get('NASA_EARTHDATA_PASSWORD')
    if not username or not password:
        raise RuntimeError('Add NASA_EARTHDATA_USERNAME and NASA_EARTHDATA_PASSWORD to Colab > Secrets and enable notebook access. Do not paste credentials into notebook cells.')
    session = requests.Session()
    login = session.post(f'{APPEEARS_API}login', auth=(username,password), timeout=60)
    if login.status_code != 200:
        raise RuntimeError(f'NASA Earthdata/AppEEARS login failed (HTTP {login.status_code}). Rotate/recheck the credentials in Colab Secrets.')
    token = login.json().get('token')
    if not token: raise RuntimeError('AppEEARS login response had no token.')
    headers = {'Authorization':f'Bearer {token}'}

    points = [{'latitude':round(float(r.Latitude),8),'longitude':round(float(r.Longitude),8),'id':str(int(r.UrbanId))}
              for r in eligible_points.itertuples(index=False)]
    chunks = [points[i:i+chunk_size] for i in range(0,len(points),chunk_size)]
    task_ids = []
    if TASK_IDS_PATH.exists():
        try:
            saved_tasks = json.loads(TASK_IDS_PATH.read_text(encoding='utf-8'))
            if isinstance(saved_tasks, dict) and saved_tasks.get('fingerprint') == request_fingerprint:
                task_ids = list(saved_tasks.get('task_ids', []))
                print(f'Resuming {len(task_ids)} AppEEARS tasks whose request fingerprint matches the current cohort.')
            else:
                print('Ignoring stale/legacy AppEEARS task IDs: request fingerprint does not match current coordinates/product/years.')
        except Exception:
            print('Ignoring unreadable AppEEARS task ID file; fresh tasks will be submitted.')
    task_ids = task_ids[:len(chunks)]

    for i, chunk in enumerate(chunks):
        task_id = task_ids[i] if i < len(task_ids) else None
        state = None
        if task_id:
            try:
                status_resp = session.get(f'{APPEEARS_API}task/{task_id}',headers=headers,timeout=60)
                if status_resp.status_code == 200: state = status_resp.json().get('status')
            except Exception:
                state = None
        if state not in ['queued','processing','done']:
            task_def = {
                'task_type':'point',
                'task_name':f'UHII_NDVI_JJA_{YEAR_MIN}_{YEAR_MAX}_batch{i+1}_{RUN_ID}',
                'params':{
                    'dates':[{'startDate':f'06-01-{y}','endDate':f'08-31-{y}'} for y in YEARS],
                    'layers':[{'product':'MOD13Q1.061','layer':'_250m_16_days_NDVI'},
                              {'product':'MOD13Q1.061','layer':'_250m_16_days_VI_Quality'}],
                    'coordinates':chunk,
                },
                'options':{'format':'csv'}
            }
            res = session.post(f'{APPEEARS_API}task',json=task_def,headers=headers,timeout=120)
            if res.status_code != 202:
                raise RuntimeError(f'AppEEARS task submit failed for batch {i+1}: HTTP {res.status_code}; {res.text[:500]}')
            task_id = res.json()['task_id']
            if i < len(task_ids): task_ids[i] = task_id
            else: task_ids.append(task_id)
            TASK_IDS_PATH.write_text(json.dumps({'fingerprint':request_fingerprint,'request_signature':request_signature,'task_ids':task_ids},indent=2),encoding='utf-8')
            print(f'Submitted NDVI batch {i+1}/{len(chunks)}; request metadata and task IDs saved.')
    if len(task_ids) != len(chunks):
        raise RuntimeError(f'AppEEARS task set incomplete: {len(task_ids)} tasks for {len(chunks)} coordinate batches.')

    deadline = time.time() + 7200
    while True:
        states = []
        for task_id in task_ids:
            rr = session.get(f'{APPEEARS_API}task/{task_id}',headers=headers,timeout=60)
            rr.raise_for_status()
            info = rr.json(); state = info.get('status')
            states.append(state)
            if state == 'error': raise RuntimeError(f'AppEEARS task failed: {task_id}; inspect the service task detail without logging credentials.')
        print('AppEEARS task states:', states)
        if all(state == 'done' for state in states): break
        if time.time() > deadline: raise TimeoutError('AppEEARS tasks are still running after 120 minutes. Re-run this cell later; matching task IDs are saved.')
        time.sleep(90)

    batch_frames = []
    for task_id in task_ids:
        br = session.get(f'{APPEEARS_API}bundle/{task_id}',headers=headers,timeout=60); br.raise_for_status()
        for f in br.json().get('files',[]):
            if not str(f.get('file_name','')).lower().endswith('.csv'): continue
            data_r = session.get(f'{APPEEARS_API}bundle/{task_id}/{f["file_id"]}',headers=headers,timeout=180)
            data_r.raise_for_status()
            batch_frames.append(pd.read_csv(BytesIO(data_r.content)))
    if not batch_frames: raise RuntimeError('No CSV observations were returned by AppEEARS.')
    raw_ndvi = pd.concat(batch_frames,ignore_index=True)
    if not {'ID','Date',NDVI_COL,QC_COL}.issubset(raw_ndvi.columns):
        raise ValueError(f'AppEEARS output lacks expected NDVI/QC columns; columns={raw_ndvi.columns.tolist()}')
    raw_ndvi['ID'] = pd.to_numeric(raw_ndvi['ID'],errors='coerce')
    raw_ndvi['Date'] = pd.to_datetime(raw_ndvi['Date'],errors='coerce')
    raw_ndvi = raw_ndvi[raw_ndvi['ID'].notna() & raw_ndvi['Date'].notna()].copy()
    raw_ndvi['ID'] = raw_ndvi['ID'].astype(int)
    if not set(raw_ndvi['ID'].unique()).issubset(expected_ids):
        unexpected = sorted(set(raw_ndvi['ID'].unique())-expected_ids)[:20]
        raise ValueError(f'AppEEARS returned IDs outside the eligible study cohort: {unexpected}')
    raw_ndvi.to_csv(NDVI_RAW_PATH,index=False)
    NDVI_METADATA_PATH.write_text(json.dumps({
        'status':'complete','request_fingerprint':request_fingerprint,'request_signature':request_signature,
        'task_ids':task_ids,'rows':int(len(raw_ndvi)),'generated_at':datetime.now().isoformat(),
        'file_sha256':file_sha256(NDVI_RAW_PATH),'file_bytes':NDVI_RAW_PATH.stat().st_size,
        'source':'NASA AppEEARS MOD13Q1.061',
    },indent=2),encoding='utf-8')
    if not valid_ndvi_raw(NDVI_RAW_PATH):
        raise RuntimeError('Fresh NDVI file failed request-fingerprint/schema/cohort/year validation; inspect AppEEARS outputs.')
    print(f'Raw NDVI saved and provenance-validated: {NDVI_RAW_PATH}; rows={len(raw_ndvi):,}; sidecar={NDVI_METADATA_PATH}')


Submitted NDVI batch 1/2; request metadata and task IDs saved.
Submitted NDVI batch 2/2; request metadata and task IDs saved.
AppEEARS task states: ['pending', 'pending']
AppEEARS task states: ['processing', 'pending']
AppEEARS task states: ['processing', 'pending']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task states: ['processing', 'processing']
AppEEARS task state

TimeoutError: AppEEARS tasks are still running after 120 minutes. Re-run this cell later; matching task IDs are saved.

Download cell timed out. Existing NDVI files were used for the analysis.

In [ ]:
from google.colab import userdata
import requests
import json

# Read the previously saved task IDs.
saved = json.loads(TASK_IDS_PATH.read_text(encoding="utf-8"))
task_ids = saved["task_ids"]

username = userdata.get("NASA_EARTHDATA_USERNAME")
password = userdata.get("NASA_EARTHDATA_PASSWORD")

session = requests.Session()
login = session.post(
    "https://appeears.earthdatacloud.nasa.gov/api/login",
    auth=(username, password),
    timeout=60
)
login.raise_for_status()

token = login.json()["token"]
headers = {"Authorization": f"Bearer {token}"}

# Check existing tasks without submitting new ones.
for i, task_id in enumerate(task_ids, 1):
    response = session.get(
        f"https://appeears.earthdatacloud.nasa.gov/api/task/{task_id}",
        headers=headers,
        timeout=60
    )
    response.raise_for_status()
    print(f"Batch {i}: {response.json().get('status')}")

Batch 1: processing
Batch 2: processing


In [ ]:
for i, task_id in enumerate(task_ids, 1):
    response = session.get(
        f"https://appeears.earthdatacloud.nasa.gov/api/task/{task_id}",
        headers=headers,
        timeout=60
    )
    response.raise_for_status()
    info = response.json()

    print(f"\nBatch {i}")
    for key in ["status", "created", "updated", "task_name", "task_type"]:
        if key in info:
            print(f"{key}: {info[key]}")


Batch 1
status: processing
created: 2026-10-18T13:05:17.353750
updated: 2026-10-18T13:05:40.335822
task_name: UHII_NDVI_JJA_2001_2020_batch1_20261018_115435
task_type: point

Batch 2
status: processing
created: 2026-10-18T13:05:17.921031
updated: 2026-10-18T13:09:35.527624
task_name: UHII_NDVI_JJA_2001_2020_batch2_20261018_115435
task_type: point


In [ ]:
from pathlib import Path
import zipfile
import shutil

# Restore the previously downloaded NASA data to the notebook's expected path.
zip_path = Path("/content/drive/MyDrive/ndvi_import_bundle.zip")
temp_dir = Path("/content/ndvi_restore_from_zip")
temp_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract("ndvi_raw_appEARS_combined.csv", temp_dir)
    z.extract("ndvi_raw_metadata.json", temp_dir)

# Preserve any existing files before restoring the imported source.
backup_dir = BACKUP_ROOT / "ndvi_source_before_restore"
backup_dir.mkdir(parents=True, exist_ok=True)

for target in [NDVI_RAW_PATH, NDVI_METADATA_PATH]:
    if target.exists():
        shutil.copy2(target, backup_dir / target.name)

shutil.copy2(
    temp_dir / "ndvi_raw_appEARS_combined.csv",
    NDVI_RAW_PATH
)
shutil.copy2(
    temp_dir / "ndvi_raw_metadata.json",
    NDVI_METADATA_PATH
)

print("NASA ham CSV dosyası yerine kondu.")
print("Kaynak:", NDVI_RAW_PATH)

NASA ham CSV dosyası yerine kondu.
Kaynak: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/data/external_features/ndvi_raw_appEARS_combined.csv


In [15]:
# Process the imported NASA AppEEARS MODIS NDVI data.

NDVI_COL = "MOD13Q1_061__250m_16_days_NDVI"
QC_COL = "MOD13Q1_061__250m_16_days_VI_Quality"

raw_ndvi = pd.read_csv(NDVI_RAW_PATH)

if not {"ID", "Date", NDVI_COL, QC_COL}.issubset(raw_ndvi.columns):
    raise ValueError(
        f"Raw NDVI schema mismatch; available columns={raw_ndvi.columns.tolist()}"
    )

raw_ndvi["ID"] = pd.to_numeric(raw_ndvi["ID"], errors="coerce")
raw_ndvi["Date"] = pd.to_datetime(raw_ndvi["Date"], errors="coerce")
raw_ndvi[NDVI_COL] = pd.to_numeric(raw_ndvi[NDVI_COL], errors="coerce")
raw_ndvi[QC_COL] = pd.to_numeric(raw_ndvi[QC_COL], errors="coerce")

raw_ndvi = raw_ndvi[
    raw_ndvi["ID"].notna() & raw_ndvi["Date"].notna()
].copy()

raw_ndvi["ID"] = raw_ndvi["ID"].astype(int)
raw_ndvi["year"] = raw_ndvi["Date"].dt.year
raw_ndvi["month"] = raw_ndvi["Date"].dt.month

# Select June, July, and August observations for 2001-2020.
jja = raw_ndvi[
    raw_ndvi["year"].between(YEAR_MIN, YEAR_MAX)
    & raw_ndvi["month"].isin([6, 7, 8])
].copy()

jja_rows_before_qc = len(jja)

# Identify duplicate city-date observations.
obs_keys = (
    jja.groupby(["ID", "Date"])
    .agg(
        n_rows=(NDVI_COL, "size"),
        n_ndvi_values=(NDVI_COL, "nunique"),
        n_qc_values=(QC_COL, "nunique")
    )
    .reset_index()
)

obs_dupes = obs_keys[obs_keys["n_rows"] > 1].copy()

obs_dupes["duplicate_type"] = np.where(
    (obs_dupes["n_ndvi_values"] <= 1)
    & (obs_dupes["n_qc_values"] <= 1),
    "exact_duplicate_observation",
    "conflicting_duplicate_observation"
)

obs_dupes.to_csv(NDVI_DUP_PATH, index=False)

conflict_keys = set(
    map(
        tuple,
        obs_dupes.loc[
            obs_dupes["duplicate_type"].eq(
                "conflicting_duplicate_observation"
            ),
            ["ID", "Date"]
        ].to_numpy()
    )
)

jja["_key"] = list(zip(jja["ID"], jja["Date"]))
jja = jja[~jja["_key"].isin(conflict_keys)].drop(columns="_key")
jja = jja.drop_duplicates(
    subset=["ID", "Date", NDVI_COL, QC_COL]
)

# Decode MODIS VI Quality flags.
def valid_mod13q1_quality(value):
    if pd.isna(value):
        return False

    q = int(value)

    modland = q & 0b11
    usefulness = (q >> 2) & 0b1111
    adjacent_cloud = (q >> 8) & 1
    mixed_cloud = (q >> 10) & 1
    snow = (q >> 14) & 1
    shadow = (q >> 15) & 1

    return (
        modland == 0
        and usefulness <= 2
        and adjacent_cloud == 0
        and mixed_cloud == 0
        and snow == 0
        and shadow == 0
    )

# The imported CSV already contains scaled NDVI values.
# Exclude fill values and retain plausible NDVI values.
jja = jja[jja[NDVI_COL].between(-0.2, 1.0)].copy()
jja = jja[jja[QC_COL].apply(valid_mod13q1_quality)].copy()

# Do not apply the 0.0001 scale factor a second time.
jja["ndvi_scaled"] = jja[NDVI_COL]

# Aggregate valid summer observations by city and year.
ndvi = (
    jja.groupby(["ID", "year"])
    .agg(
        ndvi_jja_mean=("ndvi_scaled", "mean"),
        n_ndvi_observations=("ndvi_scaled", "count")
    )
    .reset_index()
    .rename(columns={"ID": "UrbanId"})
)

# One observation is insufficient for a reliable seasonal mean.
ndvi.loc[
    ndvi["n_ndvi_observations"] < 2,
    "ndvi_jja_mean"
] = np.nan

ndvi = ndvi[
    ndvi["year"].between(YEAR_MIN, YEAR_MAX)
].copy()

if ndvi.duplicated(["UrbanId", "year"]).any():
    raise AssertionError("NDVI city-year aggregate is not unique.")

# Save the quality-filtered city-year NDVI table.
ndvi.to_csv(NDVI_QC_PATH, index=False)

# Save a concise processing summary.
qc_summary = {
    "run_id": RUN_ID,
    "raw_rows": len(raw_ndvi),
    "jja_rows_before_qc": jja_rows_before_qc,
    "rows_after_obs_dedup_and_qc": len(jja),
    "duplicate_observation_keys": len(obs_dupes),
    "conflicting_duplicate_observation_keys": int(
        obs_dupes["duplicate_type"]
        .eq("conflicting_duplicate_observation")
        .sum()
    ),
    "ndvi_city_year_rows": len(ndvi),
    "ndvi_valid_city_years": int(
        ndvi["ndvi_jja_mean"].notna().sum()
    ),
    "scale_factor_applied": 1.0,
    "raw_metadata_sidecar": str(NDVI_METADATA_PATH),
    "quality_policy": (
        "MODLAND=0, VI usefulness<=2, adjacent cloud/mixed cloud/"
        "snow/shadow flags all 0"
    )
}

pd.DataFrame([qc_summary]).to_csv(
    EXTERNAL_DIR / "ndvi_qc_summary.csv",
    index=False
)

print("NDVI kalite kontrolü tamamlandı.")
print(qc_summary)
print(f"Kaydedilen NDVI dosyası: {NDVI_QC_PATH}")
print(f"Yinelenen gözlem raporu: {NDVI_DUP_PATH}")

NDVI kalite kontrolü tamamlandı.
{'run_id': '20261019_142159', 'raw_rows': 226940, 'jja_rows_before_qc': 194520, 'rows_after_obs_dedup_and_qc': 76763, 'duplicate_observation_keys': 0, 'conflicting_duplicate_observation_keys': 0, 'ndvi_city_year_rows': 23642, 'ndvi_valid_city_years': 18237, 'scale_factor_applied': 1.0, 'raw_metadata_sidecar': '/content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/data/external_features/ndvi_raw_metadata.json', 'quality_policy': 'MODLAND=0, VI usefulness<=2, adjacent cloud/mixed cloud/snow/shadow flags all 0'}
Kaydedilen NDVI dosyası: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/data/external_features/ndvi_jja_bitmask_qc_filtered.csv
Yinelenen gözlem raporu: /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/data/external_features/ndvi_duplicate_observation_keys.csv


In [17]:
# Load previously saved feature tables; do not download anything again.
dem = pd.read_csv(DEM_PATH)
era5 = pd.read_csv(ERA5_CSV_PATH)
ndvi = pd.read_csv(NDVI_QC_PATH)

DEM_SOURCE = "Open-Meteo Elevation API / Copernicus DEM GLO-90"

with open(ERA5_METADATA_PATH, "r", encoding="utf-8") as f:
    era5_meta = json.load(f)

ERA5_REQUEST = era5_meta["request"]
ERA5_REQUEST_FINGERPRINT = era5_meta["request_fingerprint"]

print("DEM, ERA5 ve NDVI tabloları yüklendi.")
print("DEM:", dem.shape)
print("ERA5:", era5.shape)
print("NDVI:", ndvi.shape)

DEM, ERA5 ve NDVI tabloları yüklendi.
DEM: (1621, 3)
ERA5: (32420, 9)
NDVI: (23642, 4)


## Step 4 — Single and consistent external feature panel

The external property panel only uses `INCLUDED` cities in `analysis_city_manifest.csv`. The keys of the source tables are checked before merging. One row is generated for each target city-year; Deficiencies are stored and reported, they are not filled automatically.

In [18]:
# Assert source-table key uniqueness before any join.
if dem.duplicated(['UrbanId']).any(): raise ValueError('DEM table must be unique by UrbanId.')
if era5.duplicated(['UrbanId','year']).any(): raise ValueError('ERA5 table must be unique by UrbanId-year.')
if ndvi.duplicated(['UrbanId','year']).any(): raise ValueError('NDVI table must be unique by UrbanId-year.')

base_cities = eligible[['UrbanId','Country','Longitude','Latitude','Area']].copy()
years = pd.DataFrame({'year':YEARS})
external = base_cities[['UrbanId']].merge(years,how='cross')
external = external.merge(base_cities, on='UrbanId', how='left', validate='many_to_one')
external = external.merge(dem[['UrbanId','elevation_m']],on='UrbanId',how='left',validate='many_to_one')
external = external.merge(era5[['UrbanId','year','era5_t2m_jja','era5_precip_jja_mm','era5_wind_jja','era5_dewpoint_jja','era5_rh_jja','era5_months_available']],
                          on=['UrbanId','year'],how='left',validate='one_to_one')
external = external.merge(ndvi[['UrbanId','year','ndvi_jja_mean','n_ndvi_observations']],
                          on=['UrbanId','year'],how='left',validate='one_to_one')
if len(external) != len(eligible)*len(YEARS):
    raise AssertionError(f'External panel rows={len(external):,}, expected {len(eligible)*len(YEARS):,}')
if external.duplicated(['UrbanId','year']).any(): raise AssertionError('External feature panel has duplicate UrbanId-year keys.')
external.to_csv(EXTERNAL_PANEL_PATH,index=False)

feature_cols = ['elevation_m','era5_t2m_jja','era5_precip_jja_mm','era5_wind_jja','era5_dewpoint_jja','era5_rh_jja','ndvi_jja_mean']
coverage_rows=[]
for country,sub in external.groupby('Country'):
    for feature in feature_cols:
        coverage_rows.append({'Country':country,'feature':feature,'n_rows':len(sub),
                              'n_missing':int(sub[feature].isna().sum()),'missing_pct':100*sub[feature].isna().mean(),
                              'n_cities_with_any_value':int(sub.loc[sub[feature].notna(),'UrbanId'].nunique())})
coverage_df=pd.DataFrame(coverage_rows)
coverage_df.to_csv(EXTERNAL_DIR/'external_feature_coverage_by_country.csv',index=False)

summary = {
    'run_id':RUN_ID,'cities':int(external['UrbanId'].nunique()),'rows':len(external),
    'expected_rows':len(eligible)*len(YEARS),'duplicate_city_year_keys':int(external.duplicated(['UrbanId','year']).sum()),
    'era5_missing_rows':int(external['era5_t2m_jja'].isna().sum()),
    'dem_missing_cities':int(dem['elevation_m'].isna().sum()),'ndvi_missing_rows':int(external['ndvi_jja_mean'].isna().sum()),
    'source_dem':DEM_SOURCE,'era5_source_product':ERA5_REQUEST['dataset'],'era5_request_fingerprint':ERA5_REQUEST_FINGERPRINT,'era5_metadata_sidecar':str(ERA5_METADATA_PATH),'target_period':'JJA','years':f'{YEAR_MIN}-{YEAR_MAX}'
}
pd.DataFrame([summary]).to_csv(EXTERNAL_DIR/'external_features_qa_summary.csv',index=False)
report = [
    '# External Features QA Report','',f'- Run ID: `{RUN_ID}`',f'- Source cities: {summary["cities"]:,}',
    f'- Expected city-year rows: {summary["expected_rows"]:,}',f'- Actual rows: {summary["rows"]:,}',
    f'- Duplicate UrbanId-year keys: {summary["duplicate_city_year_keys"]}',
    f'- DEM missing cities: {summary["dem_missing_cities"]}',
    f'- ERA5 target-year rows missing temperature: {summary["era5_missing_rows"]:,}',
    f'- NDVI rows missing a reliable JJA mean: {summary["ndvi_missing_rows"]:,}', '',
    '## Feature coverage by country','',coverage_df.to_markdown(index=False),'',
    '## Interpretation','',
    '- Missing values are not imputed in this extraction notebook. Model-specific imputation, if scientifically appropriate, is fit only inside the training pipeline.',
    '- Open-Meteo elevation provenance is labelled as Copernicus DEM GLO-90; it is not labelled SRTM.',
    '- ERA5 precipitation is output in mm over JJA using the monthly-averaged daily-means product semantics documented by ECMWF. If the NetCDF units attribute reports only m, the conversion is allowed only after the exact CDS request fingerprint and NetCDF SHA-256 sidecar validate; otherwise the pipeline stops.',
    '- MOD13Q1 NDVI values in the imported AppEEARS CSV are already scaled; no additional 0.0001 scaling is applied. The QA policy is intentionally strict and should be documented in the study methods.',
    '- NDVI is sampled at the mapped point coordinate; it is not a polygon-weighted mean over the urban area.',
    '- ERA5 values use the nearest grid point. The wind feature is the magnitude of monthly-mean 10 m u/v vectors, not the mean scalar wind speed.',
    '- Copernicus DEM GLO-90 data access requires attribution to Copernicus and Open-Meteo.',
    '- This notebook only creates features and QA reports; it does not make a GO decision for the transfer study.'
]
(EXTERNAL_DIR/'external_features_QA_report.md').write_text('\n'.join(report),encoding='utf-8')
print(summary)
print(f'Saved {EXTERNAL_PANEL_PATH} and external-feature QA reports.')

{'run_id': '20261019_142159', 'cities': 1621, 'rows': 32420, 'expected_rows': 32420, 'duplicate_city_year_keys': 0, 'era5_missing_rows': 0, 'dem_missing_cities': 0, 'ndvi_missing_rows': 14183, 'source_dem': 'Open-Meteo Elevation API / Copernicus DEM GLO-90', 'era5_source_product': 'reanalysis-era5-land-monthly-means', 'era5_request_fingerprint': 'e9c885d1bdea9385accb7ac980037d41e51478522df4a73679ea597ae265358c', 'era5_metadata_sidecar': '/content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/outputs/phase2_outputs/era5_land_jja_2001_2020_metadata.json', 'target_period': 'JJA', 'years': '2001-2020'}
Saved /content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer/data/external_features/external_features.csv and external-feature QA reports.


In [ ]:
import pandas as pd
from pathlib import Path

# Read the external feature panel already saved to Drive.
external_path = Path(EXTERNAL_DIR) / "external_features.csv"
df = pd.read_csv(external_path)

print("Dataset shape:", df.shape)
print("Unique cities:", df["UrbanId"].nunique())
print("Duplicate city-year keys:",
      df.duplicated(["UrbanId", "year"]).sum())

# Summarize feature availability by country and year.
coverage = (
    df.groupby(["Country", "year"])
    .agg(
        city_year_rows=("UrbanId", "size"),
        era5_valid=("era5_t2m_jja", "count"),
        ndvi_valid=("ndvi_jja_mean", "count"),
    )
    .reset_index()
)

coverage["era5_missing"] = (
    coverage["city_year_rows"] - coverage["era5_valid"]
)
coverage["ndvi_missing"] = (
    coverage["city_year_rows"] - coverage["ndvi_valid"]
)

print("\n=== Coverage by country and year ===")
print(coverage.to_string(index=False))

print("\n=== Overall feature availability ===")
for feature in [
    "elevation_m",
    "era5_t2m_jja",
    "era5_precip_jja_mm",
    "era5_wind_jja",
    "era5_dewpoint_jja",
    "era5_rh_jja",
    "ndvi_jja_mean",
]:
    print(
        f"{feature}: "
        f"{df[feature].notna().sum():,}/{len(df):,} valid "
        f"({df[feature].notna().mean() * 100:.1f}%)"
    )

Dataset shape: (32420, 15)
Unique cities: 1621
Duplicate city-year keys: 0

=== Coverage by country and year ===
Country  year  city_year_rows  era5_valid  ndvi_valid  era5_missing  ndvi_missing
  China  2001            1441        1422        1031            19           410
  China  2002            1441        1422        1074            19           367
  China  2003            1441        1422         999            19           442
  China  2004            1441        1422        1048            19           393
  China  2005            1441        1422         848            19           593
  China  2006            1441        1422         922            19           519
  China  2007            1441        1422         709            19           732
  China  2008            1441        1422         778            19           663
  China  2009            1441        1422         809            19           632
  China  2010            1441        1422         629            19

In [ ]:
import json
import hashlib
import pandas as pd
from pathlib import Path

# Project paths confirmed from your Google Drive structure.
PROJECT_DIR = Path(
    "/content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer"
)

EXTERNAL_DIR = PROJECT_DIR / "data" / "external_features"
PHASE2_ROOT = PROJECT_DIR / "outputs" / "phase2_outputs"

DEM_PATH = EXTERNAL_DIR / "dem_elevation.csv"
DEM_METADATA_PATH = EXTERNAL_DIR / "dem_elevation_metadata.json"

ERA5_NC_PATH = PHASE2_ROOT / "era5_land_jja_2001_2020.nc"
ERA5_METADATA_PATH = PHASE2_ROOT / "era5_land_jja_2001_2020_metadata.json"

NDVI_RAW_PATH = EXTERNAL_DIR / "ndvi_raw_appEARS_combined.csv"
NDVI_METADATA_PATH = EXTERNAL_DIR / "ndvi_raw_metadata.json"

EXTERNAL_PANEL_PATH = EXTERNAL_DIR / "external_features.csv"


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def load_json(path):
    path = Path(path)
    if not path.is_file():
        print("MISSING FILE:", path)
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


# 1. DEM — read the saved file instead of relying on variable dem.
print("=== DEM ===")

if DEM_PATH.is_file():
    dem = pd.read_csv(DEM_PATH)
    dem_meta = load_json(DEM_METADATA_PATH)

    print("Rows:", len(dem))
    print("Unique cities:", dem["UrbanId"].nunique())
    print("Missing elevations:", dem["elevation_m"].isna().sum())
    print("Source:", dem_meta.get("source"))
    print(
        "File hash matches:",
        dem_meta.get("file_sha256") == sha256_file(DEM_PATH)
    )
else:
    print("DEM CSV not found:", DEM_PATH)


# 2. ERA5 — verify the saved NetCDF and its provenance.
print("\n=== ERA5-Land ===")

era5_meta = load_json(ERA5_METADATA_PATH)

print("NetCDF exists:", ERA5_NC_PATH.is_file())

if era5_meta and ERA5_NC_PATH.is_file():
    actual_hash = sha256_file(ERA5_NC_PATH)

    print("Source product:", era5_meta.get("source_product"))
    print(
        "Metadata hash matches:",
        era5_meta.get("file_sha256") == actual_hash
    )
    print(
        "Request fingerprint recorded:",
        bool(era5_meta.get("request_fingerprint"))
    )

    expected_fp = globals().get("ERA5_REQUEST_FINGERPRINT")
    if expected_fp:
        print(
            "Current request fingerprint matches:",
            era5_meta.get("request_fingerprint") == expected_fp
        )


# 3. External feature panel — load the saved CSV directly.
print("\n=== EXTERNAL FEATURES ===")

if not EXTERNAL_PANEL_PATH.is_file():
    raise FileNotFoundError(
        f"External feature panel not found: {EXTERNAL_PANEL_PATH}"
    )

external = pd.read_csv(EXTERNAL_PANEL_PATH)

print("Shape:", external.shape)
print("Unique cities:", external["UrbanId"].nunique())
print(
    "Duplicate city-year keys:",
    external.duplicated(["UrbanId", "year"]).sum()
)
print("Missing DEM rows:", external["elevation_m"].isna().sum())
print("Missing ERA5 temperature rows:",
      external["era5_t2m_jja"].isna().sum())
print("Missing NDVI rows:", external["ndvi_jja_mean"].isna().sum())

# Identify cities with no valid ERA5 temperature across the panel.
era5_missing = (
    external.loc[
        external["era5_t2m_jja"].isna(),
        ["UrbanId", "Country", "Latitude", "Longitude"]
    ]
    .drop_duplicates("UrbanId")
    .sort_values(["Country", "UrbanId"])
)

print("\nCities missing ERA5 temperature:", len(era5_missing))
print(era5_missing.to_string(index=False))


# 4. NASA NDVI — verify the imported raw CSV and metadata.
print("\n=== NASA NDVI ===")

print("Raw CSV exists:", NDVI_RAW_PATH.is_file())
print("Metadata exists:", NDVI_METADATA_PATH.is_file())

if NDVI_RAW_PATH.is_file() and NDVI_METADATA_PATH.is_file():
    ndvi_meta = load_json(NDVI_METADATA_PATH)

    print("Source:", ndvi_meta.get("source"))
    print("Status:", ndvi_meta.get("status"))
    print(
        "File hash matches:",
        ndvi_meta.get("file_sha256") == sha256_file(NDVI_RAW_PATH)
    )
    print(
        "Request fingerprint recorded:",
        bool(ndvi_meta.get("request_fingerprint"))
    )

=== DEM ===
Rows: 1621
Unique cities: 1621
Missing elevations: 0
Source: Open-Meteo Elevation API / Copernicus DEM GLO-90
File hash matches: True

=== ERA5-Land ===
NetCDF exists: True
Source product: reanalysis-era5-land-monthly-means
Metadata hash matches: True
Request fingerprint recorded: True

=== EXTERNAL FEATURES ===
Shape: (32420, 15)
Unique cities: 1621
Duplicate city-year keys: 0
Missing DEM rows: 0
Missing ERA5 temperature rows: 680
Missing NDVI rows: 14183

Cities missing ERA5 temperature: 34
 UrbanId Country  Latitude  Longitude
     364   China 39.935474 119.604444
     903   China 21.239554 110.369411
    2013   China 30.731153 121.310107
    2781   China 39.417894 121.681932
    4218   China 40.935359 121.225583
    4774   China 39.235002 118.185821
    5486   China 39.229810 119.000950
    6313   China 22.825222 115.830471
    6773   China 25.430095 119.732858
    7176   China 31.632873 121.399062
    7272   China 18.408857 109.864572
    7589   China 25.216669 119.469

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer"
)

PHASE2_ROOT = PROJECT_DIR / "outputs" / "phase2_outputs"
EXTERNAL_DIR = PROJECT_DIR / "data" / "external_features"

nc_path = PHASE2_ROOT / "era5_land_jja_2001_2020.nc"
panel_path = EXTERNAL_DIR / "external_features.csv"

external = pd.read_csv(panel_path)
missing_cities = (
    external.loc[
        external["era5_t2m_jja"].isna(),
        ["UrbanId", "Country", "Latitude", "Longitude"]
    ]
    .drop_duplicates("UrbanId")
    .sort_values(["Country", "UrbanId"])
)

ds = xr.open_dataset(nc_path)

time_name = next(c for c in ds.coords if "time" in c.lower())
lat_name = "latitude" if "latitude" in ds.coords else "lat"
lon_name = "longitude" if "longitude" in ds.coords else "lon"

lats = np.asarray(ds[lat_name].values, dtype=float)
lons = np.asarray(ds[lon_name].values, dtype=float)

times = pd.to_datetime(ds[time_name].values)
jja_idx = np.flatnonzero(pd.DatetimeIndex(times).month.isin([6, 7, 8]))

print("NetCDF dimensions:", dict(ds.sizes))
print("NetCDF latitude range:", float(lats.min()), "to", float(lats.max()))
print("NetCDF longitude range:", float(lons.min()), "to", float(lons.max()))
print("JJA time points:", len(jja_idx))
print("Cities to investigate:", len(missing_cities))

variables = ["t2m", "tp", "u10", "v10", "d2m"]
results = []

for city in missing_cities.itertuples(index=False):
    lat, lon = float(city.Latitude), float(city.Longitude)

    # Find the nearest grid cell using coordinate distance.
    i = int(np.abs(lats - lat).argmin())
    lon_diff = np.abs((lons - lon + 180) % 360 - 180)
    j = int(lon_diff.argmin())

    row = {
        "UrbanId": city.UrbanId,
        "Country": city.Country,
        "grid_lat": float(lats[i]),
        "grid_lon": float(lons[j]),
        "grid_t2m_valid": 0,
        "max_t2m_valid_in_3x3": 0,
    }

    # Count valid JJA values at the exact cell for every required variable.
    for var in variables:
        point = ds[var].isel({lat_name: i, lon_name: j}).squeeze(drop=True)
        if time_name in point.dims:
            point = point.isel({time_name: jja_idx})
            row[f"{var}_valid"] = int(np.isfinite(point.values).sum())
        else:
            row[f"{var}_valid"] = 0

    row["grid_t2m_valid"] = row["t2m_valid"]

    # Check the nearest cell and its immediate 3x3 neighborhood.
    i0, i1 = max(0, i - 1), min(len(lats), i + 2)
    j0, j1 = max(0, j - 1), min(len(lons), j + 2)

    window = ds["t2m"].isel({
        lat_name: slice(i0, i1),
        lon_name: slice(j0, j1),
    }).squeeze(drop=True)

    window = window.isel({time_name: jja_idx})
    valid_by_cell = window.notnull().sum(time_name)

    row["max_t2m_valid_in_3x3"] = int(valid_by_cell.max().values)

    results.append(row)

diagnostic = pd.DataFrame(results)

print("\n=== ERA5 missing-city diagnosis ===")
print(diagnostic.to_string(index=False))

ds.close()

NetCDF dimensions: {'valid_time': 60, 'latitude': 371, 'longitude': 1101}
NetCDF latitude range: 18.0 to 55.0
NetCDF longitude range: 25.0 to 135.0
JJA time points: 60
Cities to investigate: 34

=== ERA5 missing-city diagnosis ===
 UrbanId Country  grid_lat  grid_lon  grid_t2m_valid  max_t2m_valid_in_3x3  t2m_valid  tp_valid  u10_valid  v10_valid  d2m_valid
     364   China      39.9     119.6               0                    60          0         0          0          0          0
     903   China      21.2     110.4               0                    60          0         0          0          0          0
    2013   China      30.7     121.3               0                    60          0         0          0          0          0
    2781   China      39.4     121.7               0                    60          0         0          0          0          0
    4218   China      40.9     121.2               0                    60          0         0          0          0       

In [ ]:
# Diagnose the nearest complete ERA5-Land grid cell.
# This cell does not modify or download any files.

import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Urban_Heat_Transfer_COMPLETE_BUNDLE/urban-heat-transfer"
)
NC_PATH = PROJECT_DIR / "outputs/phase2_outputs/era5_land_jja_2001_2020.nc"
PANEL_PATH = PROJECT_DIR / "data/external_features/external_features.csv"

external = pd.read_csv(PANEL_PATH)
missing = (
    external.loc[
        external["era5_t2m_jja"].isna(),
        ["UrbanId", "Country", "Latitude", "Longitude"]
    ]
    .drop_duplicates("UrbanId")
    .reset_index(drop=True)
)

ds = xr.open_dataset(NC_PATH)

time_name = "valid_time" if "valid_time" in ds.coords else "time"
lat_name = "latitude" if "latitude" in ds.coords else "lat"
lon_name = "longitude" if "longitude" in ds.coords else "lon"

lats = ds[lat_name].values.astype(float)
lons = ds[lon_name].values.astype(float)

# A grid cell is eligible only if all five variables
# have all 60 JJA monthly observations.
variables = ["t2m", "tp", "u10", "v10", "d2m"]
complete = np.ones((len(lats), len(lons)), dtype=bool)

for var in variables:
    values = ds[var].transpose(
        time_name, lat_name, lon_name
    ).values.astype(float)

    valid_count = np.isfinite(values).sum(axis=0)
    complete &= valid_count == len(ds[time_name])

grid_lat, grid_lon = np.meshgrid(lats, lons, indexing="ij")
valid_i, valid_j = np.where(complete)

candidate_lat = grid_lat[valid_i, valid_j]
candidate_lon = grid_lon[valid_i, valid_j]

print("Complete grid cells:", len(valid_i))
print("Cities to investigate:", len(missing))

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 6371.0 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

results = []

for city in missing.itertuples(index=False):
    distances = haversine_km(
        city.Latitude, city.Longitude,
        candidate_lat, candidate_lon
    )

    k = int(np.argmin(distances))

    results.append({
        "UrbanId": city.UrbanId,
        "Country": city.Country,
        "city_lat": city.Latitude,
        "city_lon": city.Longitude,
        "nearest_valid_lat": candidate_lat[k],
        "nearest_valid_lon": candidate_lon[k],
        "distance_km": round(float(distances[k]), 2),
    })

audit = pd.DataFrame(results).sort_values("distance_km")

print("\n=== Nearest complete-cell audit ===")
print(audit.to_string(index=False))

print("\nCities within 15 km:", int((audit["distance_km"] <= 15).sum()))
print("Cities within 30 km:", int((audit["distance_km"] <= 30).sum()))
print("Maximum distance (km):", audit["distance_km"].max())

ds.close()

Complete grid cells: 343265
Cities to investigate: 34

=== Nearest complete-cell audit ===
 UrbanId Country  city_lat   city_lon  nearest_valid_lat  nearest_valid_lon  distance_km
    3956 Türkiye 36.603216  31.843444               36.6               31.9         5.06
    8144   China 26.496947 119.550821               26.5              119.5         5.07
    3568 Türkiye 38.774377  26.946450               38.8               27.0         5.45
     261 Türkiye 38.427447  27.144654               38.4               27.2         5.71
    5498 Türkiye 36.548502  32.017158               36.6               32.0         5.93
    6193 Türkiye 36.245078  30.003022               36.3               30.0         6.11
    8380 Türkiye 36.110940  32.967179               36.1               32.9         6.16
    2655 Türkiye 41.460532  31.809158               41.4               31.8         6.77
    4759 Türkiye 41.270566  31.427814               41.3               31.5         6.86
    9691   China 32

In [19]:

# Check NDVI missingness by country and year.

ndvi_missing = external.assign(
    ndvi_missing=external["ndvi_jja_mean"].isna()
)

print("NDVI coverage by country:")
print(
    ndvi_missing.groupby("Country")["ndvi_missing"]
    .agg(["count", "sum", "mean"])
    .assign(missing_pct=lambda x: x["mean"] * 100)
    .drop(columns="mean")
    .round(2)
)

print("\nNDVI missingness by year:")
print(
    ndvi_missing.groupby("year")["ndvi_missing"]
    .agg(["count", "sum", "mean"])
    .assign(missing_pct=lambda x: x["mean"] * 100)
    .drop(columns="mean")
    .round(2)
)


NDVI coverage by country:
         count    sum  missing_pct
Country                           
China    28820  13928        48.33
Türkiye   3600    255         7.08

NDVI missingness by year:
      count  sum  missing_pct
year                         
2001   1621  417        25.72
2002   1621  376        23.20
2003   1621  448        27.64
2004   1621  399        24.61
2005   1621  599        36.95
2006   1621  527        32.51
2007   1621  739        45.59
2008   1621  669        41.27
2009   1621  637        39.30
2010   1621  821        50.65
2011   1621  830        51.20
2012   1621  829        51.14
2013   1621  733        45.22
2014   1621  846        52.19
2015   1621  919        56.69
2016   1621  786        48.49
2017   1621  920        56.76
2018   1621  856        52.81
2019   1621  896        55.27
2020   1621  936        57.74


In [20]:

# Compare NDVI coverage before and after quality filtering.

print("NDVI columns:")
print(ndvi.columns.tolist())

print("\nNDVI table shape:", ndvi.shape)

print("\nNDVI missing values by column:")
print(ndvi.isna().sum())

print("\nNDVI valid observations per year:")
print(
    ndvi.groupby("year")["ndvi_jja_mean"]
    .agg(["count", "mean", "min", "max"])
    .round(3)
)


NDVI columns:
['UrbanId', 'year', 'ndvi_jja_mean', 'n_ndvi_observations']

NDVI table shape: (23642, 4)

NDVI missing values by column:
UrbanId                   0
year                      0
ndvi_jja_mean          5405
n_ndvi_observations       0
dtype: int64

NDVI valid observations per year:
      count   mean    min    max
year                            
2001   1204  0.356 -0.021  0.873
2002   1245  0.349  0.018  0.839
2003   1173  0.350  0.018  0.824
2004   1222  0.354  0.015  0.836
2005   1022  0.344 -0.040  0.851
2006   1094  0.342 -0.022  0.868
2007    882  0.342 -0.020  0.887
2008    952  0.356  0.033  0.857
2009    984  0.337 -0.053  0.868
2010    800  0.346  0.010  0.855
2011    791  0.344 -0.106  0.853
2012    792  0.351 -0.095  0.869
2013    888  0.362 -0.035  0.852
2014    775  0.346  0.080  0.875
2015    702  0.359  0.005  0.879
2016    835  0.363 -0.139  0.868
2017    701  0.386  0.071  0.876
2018    765  0.369 -0.124  0.859
2019    725  0.378  0.071  0.871
2020    685

In [21]:

# Investigate missing NDVI values and observation counts.

print("Observation counts for rows with missing NDVI:")
print(
    ndvi.loc[
        ndvi["ndvi_jja_mean"].isna(),
        "n_ndvi_observations"
    ].describe()
)

print("\nObservation counts for rows with valid NDVI:")
print(
    ndvi.loc[
        ndvi["ndvi_jja_mean"].notna(),
        "n_ndvi_observations"
    ].describe()
)

print("\nMissing NDVI by country:")
ndvi_check = ndvi.merge(
    eligible[["UrbanId", "Country"]],
    on="UrbanId",
    how="left",
    validate="many_to_one"
)

print(
    ndvi_check.assign(
        missing=ndvi_check["ndvi_jja_mean"].isna()
    )
    .groupby("Country")["missing"]
    .agg(["count", "sum", "mean"])
    .assign(missing_pct=lambda x: x["mean"] * 100)
    .drop(columns="mean")
    .round(2)
)


Observation counts for rows with missing NDVI:
count    5405.0
mean        1.0
std         0.0
min         1.0
25%         1.0
50%         1.0
75%         1.0
max         1.0
Name: n_ndvi_observations, dtype: float64

Observation counts for rows with valid NDVI:
count    18237.000000
mean         3.912815
std          1.485792
min          2.000000
25%          3.000000
50%          4.000000
75%          5.000000
max          6.000000
Name: n_ndvi_observations, dtype: float64

Missing NDVI by country:
         count   sum  missing_pct
Country                          
China    20219  5327        26.35
Türkiye   3423    78         2.28


In [22]:

# Check the filtered NDVI data.

ndvi_obs = pd.read_csv(NDVI_QC_PATH)

print("Filtered NDVI data:")
print("Rows:", len(ndvi_obs))
print("Columns:", ndvi_obs.columns.tolist())

print("\nFirst rows:")
print(ndvi_obs.head())

print("\nAvailable observations by city and year:")
print(
    ndvi_obs.groupby(["UrbanId", "year"])
    .size()
    .describe()
)


Filtered NDVI data:
Rows: 23642
Columns: ['UrbanId', 'year', 'ndvi_jja_mean', 'n_ndvi_observations']

First rows:
   UrbanId  year  ndvi_jja_mean  n_ndvi_observations
0        5  2001       0.480600                    3
1        5  2002       0.557700                    5
2        5  2003       0.476500                    3
3        5  2004       0.411900                    2
4        5  2005       0.427167                    3

Available observations by city and year:
count    23642.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
dtype: float64


In [23]:

# Check the NDVI data status.

ndvi_status = external.copy()

ndvi_status["ndvi_status"] = "No valid observation after filtering"

ndvi_status.loc[
    ndvi_status["ndvi_jja_mean"].notna(),
    "ndvi_status"
] = "Valid NDVI"

ndvi_status.loc[
    ndvi_status["ndvi_jja_mean"].isna()
    & ndvi_status["n_ndvi_observations"].eq(1),
    "ndvi_status"
] = "One observation"

print("NDVI status by country:")
print(
    ndvi_status.groupby(
        ["Country", "ndvi_status"]
    ).size().unstack(fill_value=0)
)

print("\nTotal:")
print(ndvi_status["ndvi_status"].value_counts())


NDVI status by country:
ndvi_status  No valid observation after filtering  One observation  Valid NDVI
Country                                                                       
China                                        8601             5327       14892
Türkiye                                       177               78        3345

Total:
ndvi_status
Valid NDVI                              18237
No valid observation after filtering     8778
One observation                          5405
Name: count, dtype: int64


In [24]:

# Save the NDVI coverage summary.

coverage = (
    ndvi_status.groupby(["Country", "ndvi_status"])
    .size()
    .unstack(fill_value=0)
)

coverage["total_city_years"] = coverage.sum(axis=1)

for col in [
    "Valid NDVI",
    "One observation",
    "No valid observation after filtering"
]:
    if col not in coverage.columns:
        coverage[col] = 0

coverage["valid_ndvi_pct"] = (
    coverage["Valid NDVI"] / coverage["total_city_years"] * 100
).round(2)

coverage["missing_ndvi_pct"] = (
    100 - coverage["valid_ndvi_pct"]
).round(2)

coverage.to_csv(
    EXTERNAL_DIR / "ndvi_coverage_summary.csv"
)

print(coverage)
print("\nNDVI coverage summary saved.")


ndvi_status  No valid observation after filtering  One observation  \
Country                                                              
China                                        8601             5327   
Türkiye                                       177               78   

ndvi_status  Valid NDVI  total_city_years  valid_ndvi_pct  missing_ndvi_pct  
Country                                                                      
China             14892             28820           51.67             48.33  
Türkiye            3345              3600           92.92              7.08  

NDVI coverage summary saved.


In [25]:

# Check the final external feature panel.

panel = pd.read_csv(EXTERNAL_PANEL_PATH)

print("Rows:", len(panel))
print("Cities:", panel["UrbanId"].nunique())
print("Years:", panel["year"].nunique())
print(
    "Duplicate city-year rows:",
    panel.duplicated(["UrbanId", "year"]).sum()
)

print("\nMissing values:")
print(
    panel[
        [
            "elevation_m",
            "era5_t2m_jja",
            "era5_precip_jja_mm",
            "era5_wind_jja",
            "era5_dewpoint_jja",
            "era5_rh_jja",
            "ndvi_jja_mean",
        ]
    ].isna().sum()
)

print("\nNotebook 2 output files:")
for path in [
    EXTERNAL_PANEL_PATH,
    EXTERNAL_DIR / "external_feature_coverage_by_country.csv",
    EXTERNAL_DIR / "external_features_qa_summary.csv",
    EXTERNAL_DIR / "external_features_QA_report.md",
    EXTERNAL_DIR / "ndvi_coverage_summary.csv",
]:
    print(path.name, "OK" if path.exists() else "MISSING")


Rows: 32420
Cities: 1621
Years: 20
Duplicate city-year rows: 0

Missing values:
elevation_m               0
era5_t2m_jja              0
era5_precip_jja_mm        0
era5_wind_jja             0
era5_dewpoint_jja         0
era5_rh_jja               0
ndvi_jja_mean         14183
dtype: int64

Notebook 2 output files:
external_features.csv OK
external_feature_coverage_by_country.csv OK
external_features_qa_summary.csv OK
external_features_QA_report.md OK
ndvi_coverage_summary.csv OK
